<div style="border: 2px solid #8A9AD0; margin: 1em 0.2em; padding: 0.5em;">

# Filter, plot and explore single-cell RNA-seq data with Scanpy (Python)

by [Morgan Howells](https://training.galaxyproject.org/hall-of-fame/hexhowells/), [Wendi Bacon](https://training.galaxyproject.org/hall-of-fame/nomadscientist/)

CC-BY licensed content from the [Galaxy Training Network](https://training.galaxyproject.org/)

**Objectives**

- Is my single cell dataset a quality dataset?
- How do I generate and annotate cell clusters?
- How do I pick thresholds and parameters in my analysis? What's a "reasonable" number, and will the world collapse if I pick the wrong one?

**Objectives**

- Interpret quality control plots to direct parameter decisions
- Repeat analysis from matrix to clustering
- Identify decision-making points
- Appraise data outputs and decisions
- Explain why single cell analysis is an iterative (i.e. the first plots you generate are not final, but rather you go back and re-analyse your data repeatedly) process

**Time Estimation: 3H**
</div>


<h1 id="install-libraries">Install libraries</h1>
<p>This tutorial requies some libraries to be installed which is done below (igraph and louvain are not used directly and are just required for plotting). The <code class="language-plaintext highlighter-rouge">-q</code> parameter hides most of the outputs of the installation in order to make the notebook a bit cleaner. If there are any issues with the installation, then removing this parameter may give you more information about the issue.</p>


In [ ]:
pip install scanpy -q

In [ ]:
pip install igraph -q

In [ ]:
pip install louvain -q

In [ ]:
pip install pandas -q

<hr />
<p>We can now import the two libraries that we will be using, <strong>scanpy</strong> is the primary library that we will use and will handle all the plotting and data processing. Meanwhile, <strong>pandas</strong> is used briefly for some manual data manipulation.</p>


In [ ]:
import scanpy as sc
import pandas as pd

<h1 id="load-data">Load Data</h1>
<p>You can import files from your Galaxy history directly using the following code. This will depend on what number in your history the final annotated object is. If your object is dataset #1 in your history, then you import it with the following:</p>


In [ ]:
mito_counted_anndata = get(1)                   # get an object from Galaxy history
adata = sc.read_h5ad(mito_counted_anndata)      # read in the file as h5ad object

<p>Alternatively, if you don’t want to get the dataset from your Galaxy history, you can also download the input file from Zenodo, running the code below:</p>


In [ ]:
%%bash
wget -nv https://zenodo.org/record/7053673/files/Mito-counted_AnnData

In [ ]:
adata = sc.read_h5ad("Mito-counted_AnnData")

<h1 id="filtering">Filtering</h1>
<p>You have generated an annotated AnnData object from your raw scRNA-seq fastq files. However, you have only completed a ‘rough’ filter of your dataset - there will still be a number of ‘cells’ that are actually just background from empty droplets or simply low-quality. There will also be genes that could be sequencing artifacts or that appear with such low frequency that statistical tools will fail to analyse them. This background garbage of both cells and genes not only makes it harder to distinguish real biological information from the noise, but also makes it computationally heavy to analyse. These spurious reads take a lot of computational power to analyse! First on our agenda is to filter this matrix to give us cleaner data to extract meaningful insight from, and to allow faster analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>What information is stored in your AnnData object? The last tool to generate this object counted the mitochondrial associated genes in your matrix. Where is that data stored?</li>
<li>While you are figuring that out, how many genes and cells are in your object?</li>
</ol>
<blockquote class="tip" style="border: 2px solid #FFE19E; margin: 1em 0.2em">
<div class="box-title tip-title" id="tip-hint"><button class="gtn-boxify-button tip" type="button" aria-controls="tip-hint" aria-expanded="true"><i class="far fa-lightbulb" aria-hidden="true" ></i> <span>Tip: Hint</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Inspect the Anndata object by printing it with:</p>
<div class="language-plaintext highlighter-rouge"><div><pre style="color: inherit; background: transparent"><code style="color: inherit">print(adata)

print(adata.obs)

print(adata.var)
</code></pre></div>    </div>
</blockquote>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution"><button class="gtn-boxify-button solution" type="button" aria-controls="solution" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>If you examine your AnnData object, you’ll find a number of different quality control metrics for both cells (<strong>obs</strong>) and genes (<strong>var</strong>).
<ul>
<li>For instance, you can see a <code style="color: inherit">n_cells</code> under <strong>var</strong>, which counts the number of cells that gene appears in.</li>
<li>In the <strong>obs</strong>, you have both discrete and log-based metrics for <code style="color: inherit">n_genes</code>, how many genes are counted in a cell, and <code style="color: inherit">n_counts</code>, how many UMIs are counted per cell. So, for instance, you might count multiple GAPDHs in a cell. Your <code style="color: inherit">n_counts</code> should thus be higher than <code style="color: inherit">n_genes</code>.</li>
<li>But what about the mitochondria?? Within the cells information <strong>obs</strong>, the <code style="color: inherit">total_counts_mito</code>,  <code style="color: inherit">log1p_total_counts_mito</code>, and <code style="color: inherit">pct_counts_mito</code> has been calculated for each cell.</li>
</ul>
</li>
<li>You can see by printing the object that the matrix is <code style="color: inherit">31178 x 35734</code>. This is <code style="color: inherit">obs x vars</code>, or rather, <code style="color: inherit">cells x genes</code>, so there are <code style="color: inherit">31178 cells</code> and <code style="color: inherit">35734 genes</code> in the matrix.</li>
</ol>
</details>
</blockquote>
<h2 id="generate-qc-plots">Generate QC Plots</h2>
<p>We want to filter our cells, but first we need to know what our data looks like. There are a number of subjective choices to make within scRNA-seq analysis, for instance we now need to make our best informed decisions about where to set our thresholds (more on that soon!). We’re going to plot our data a few different ways. Different bioinformaticians might prefer to see the data in different ways, and here we are only generating some of the myriad of plots you can use. Ultimately you need to go with what makes the most sense to you.</p>
<h2 id="creating-the-plots">Creating the Plots</h2>


In [ ]:
# Violin - genotype - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-genotype-log.png'
)

In [ ]:
# Violin - sex - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='sex',
  save='-sex-log.png'
)

In [ ]:
# Violin - batch - log
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='batch',
  save='-batch-log.png'
)

In [ ]:
# Scatter - mito x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='pct_counts_mito',
  save='-mitoxUMIs.png'
)

In [ ]:
# Scatter - mito x genes
sc.pl.scatter(
  adata,
  x='log1p_n_genes_by_counts',
  y='pct_counts_mito',
  save='-mitoxgenes.png'
)

In [ ]:
# Scatter - genes x UMIs
sc.pl.scatter(
  adata,
  x='log1p_total_counts',
  y='log1p_n_genes_by_counts',
  color='pct_counts_mito',
  save='-genesxUMIs.png'
)

<h2 id="analysing-the-plots">Analysing the plots</h2>
<p>That’s a lot of information! Let’s attack this in sections and see what questions these plots can help us answer.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-variation"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch Variation</div>
<p>Are there differences in sequencing depth across the samples?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret it?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-1"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-1" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>The plot <code style="color: inherit">violin - batch - log</code> will have what you’re looking for!
<figure id="figure-1" style="max-width: 90%;"><img src="...
QmCC
"" alt="Violin - batch - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-batch-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 1</strong>:</span> Violin - batch - log (Raw)</figcaption></figure>
</li>
<li>Keeping in mind that this is a log scale - which means that small differences can mean large differences - the violin plots probably look pretty similar.
<ul>
<li><code style="color: inherit">N703</code> and <code style="color: inherit">N707</code> might be a bit lower on genes and counts (or UMIs), but the differences aren’t catastrophic.</li>
<li>The <code style="color: inherit">pct_counts_mito</code> looks pretty similar across the batches, so this also looks good.</li>
<li>Nothing here would cause us to eliminate a sample from our analysis, but if you see a sample looking completely different from the rest, you would need to question why that is and consider eliminating it from your experiment!</li>
</ul>
</li>
</ol>
</details>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-biological-variables"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Biological Variables</div>
<p>Are there differences in sequencing depth across sex? Genotype?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>How do you interpret the <code style="color: inherit">sex</code> differences?</li>
<li>How do you interpret the <code style="color: inherit">genotype</code> differences?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-2"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-2" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Similar to above, the plots <code style="color: inherit">violin - sex - log</code> and <code style="color: inherit">violin - genotype - log</code> will have what you’re looking for!
<figure id="figure-2" style="max-width: 90%;"><img src="...
AAAASUVORK5CYII=
"" alt="Violin - sex - log. " width="1339" height="255" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-sex-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 2</strong>:</span> Violin - sex - log (Raw)</figcaption></figure>
<figure id="figure-3" style="max-width: 90%;"><img src="...
/wdHQxKkd1E9GQAAAABJRU5ErkJggg==
"" alt="Violin - genotype - log. " width="1453" height="297" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-genotype-log.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 3</strong>:</span> Violin - genotype - log (Raw)</figcaption></figure>
</li>
<li>There isn’t a major difference in sequencing depth across sex, I would say - though you are welcome to disagree!
<ul>
<li>It is clear there are far fewer female cells, which makes sense given that only one sample was female. <em>Note - that was an unfortunate discovery made long after generating libraries. It’s quite hard to identify the sex of a neonate in the lab! In practice, try hard to not let such a confounding factor into your data! You could consider re-running all the following analysis without that female sample, if you wish.</em></li>
</ul>
</li>
<li>In <code style="color: inherit">Violin - genotype - log</code>, however, we can see there is a difference. The <code style="color: inherit">knockout</code> samples clearly have fewer genes and counts. From an experimental point of view, we can consider, does this make sense?
<ul>
<li>Would we biologically expect that those cells would be smaller or having fewer transcripts? Possibly, in this case, given that these cells were generated by growth restricted neonatal mice, and in which case we don’t need to worry about our good data, but rather keep this in mind when generating clusters, as we don’t want depth to define clusters, we want biology to!</li>
<li>On the other hand, it may be that those cells didn’t survive dissociation as well as the healthy ones (in which case we’d expect higher mitochondrial-associated genes, which we don’t see, so we can rule that out!).</li>
<li>Maybe we unluckily poorly prepared libraries for specifically those knockout samples. There are only three, so maybe those samples are under-sequenced.</li>
<li>So what do we do about all of this?
<ul>
<li>Ideally, we consider re-sequencing all the samples but with a higher concentration of the knockout samples in the library. Any bioinformatician will tell you that the best way to get clean data is in the lab, not the computer! Sadly, absolute best practice isn’t necessarily always a realistic option in the lab - for instance, that mouse line was long gone! - so sometimes, we have to make the best of it. There are options to try and address such discrepancy in sequencing depth. Thus, we’re going to take these samples forward and see if we can find biological insight despite the technical differences.</li>
</ul>
</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<p>Now that we’ve assessed the differences in our samples, we will look at the libraries overall to identify appropriate thresholds for our analysis.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-genes"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: genes</div>
<p>What threshold should you set for <code style="color: inherit">log1p_n_genes_by_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-3"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-3" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">log1p_n_genes_by_counts</code> would do here, actually! Some people prefer scatterplots to violins.
<figure id="figure-4" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAASFBMVEX////z
8/OgoKCEhIT4+PiLi4vZ2dmAgID8/PwzMzO0tLR4eHjt7e3n5+fh4eHGxsaV
lZW+vr7Ozs6qqqpWVlYWFhYrKys/Pz+hzycdAAAACXBIWXMAAA7zAAAO8wEc
U5k6AAAgAElEQVR42u1diXbrKAw1GBOMAa9J/v9P50rgLXG2Np3XxT4zfW3q
uC1C0tXVQpa9dKlsv774Upl59S1mX7X/4zIfkOV+fbm6vLzOu1T+R9G8cIl9
yb6fi9mV5f8Syqvuxe9r9j9c4lUNEKdTuV9ffh3FNiIzN3Fa+RGwt0PrF6/y
Zb+yy+Vfy8WYLT9f7jHov9YXkz4AG8z7vsx2fuAfysWs7Jcy02vlvmj/Xi6K
/1ejVfuIXHZteb9cjDsfy0N2OJ5KO67wi3IxahfMm+ViMhXqTJ/bolhs/Bfl
otROEbxRLmYSTXZuq2LBcZ5fs0yuE+tH7tc74heX2TLkx1MVWCjyNBxfpznN
7mzepi+jW+gGmXmXVdX4+jn7P1i4XS7340oxHPjzcBy3++kjeEwLtSvLG+2Y
GkhPnMqKatz4H9IXv8vlnXJx+ngsSzmUp0Hz1n893t8JsnfLxUzh/grpnl5d
ZrP7/S/gkxfcpfpQvD/JYsfJX8Dzf+Yd+7XLZZfLfu1y2eWyy2WXy37tctnl
sl+7XHa57HLZ5bJfu1x2uezXLpddLvu1y2WXyy6XXS7ZnsXc9WWXy64tP19f
9tLybykX4/fe82+pL87tgviW/mV3Ma/2V3IHzOq10z6O5Fv0I+84+bvaMbMS
0i6XbyCXqVtc7Pryber54Vr8+XSuMj8ch25sFD/vi/bP9UU5LfxQV42p8l1f
vlO/GDUi1SedhTI1fKldX76H39cl91bGNuQP9CPv1/vlAgXx5z6b5KLeZ8dM
EDcx+QwAzV8lNh/271PjK9mxU/bheUrbVGVwlwtuzKWElNC7vmz17wsSi6ia
rDh8cA7JzUuYO2HsKCLhd7lsrZ3GxJ5jrc+Mk+NSll85XFuovRfzGX5smn6J
Wb7infqiNn+k8G7v83+4ymzK1OV05S+KXwyZLuPW1s38Wb0pHwx0uXLFb5WL
MVufrr70u748RyiXX5PVN6N2rEf8GCuyEMwul3e94wk/oe4OO12AZaF2ubzp
Heqp8Vbmxitro3Y5itGYXS7ZJyZhmztpTbUaG7Nf/2d+3z0wZeaJl7d1w+xy
+eA7oA3OigdSe0Zi5koyateXz73jvlj0jfUVYh/L+JV4TN3f2upGYKJCUI80
CffscvngOx64dXUbSJvHkY8Tf08u4oEJL//v0wL/JHArXz5ArPwJJzn+OrmY
zFXnsnLfSi5ql0uWDXmni+H7yMXscuFVKLP7w6vL3Yz9C30RZ5woUpffSy47
n5xl7RmzrLvv1V8ZdrlkwTgUq3wjuUBdarXLpXxQJFY+Y3SuJ4y/bIzCvbeo
vyYX3R/73jaflcuaD3av4ypziz8zY2Lzb8nFDsdqqCr7qcnx5qJ6cqwVCK9s
c3enJuAv8vztA2RavgRsVeJ2uLRGi5ew8ZZauD/qXwpF6lINn+pk8mq1ksa0
jiUknt/ot3yL9X+j2//KjmVWStnLz+iL0ItdrVqz2OWvGiCz68vNepTX7Zi7
8YX5UGXZnxyOcb3K9XAqT5/gYcyqt1y3bi602CstPiGXY6uD15+Sy1z5bTpv
XpidYKbGzr+eoLnOvwxhdT7y1QqdXvAKZszxG3NfrZbnMZM/sld1MTcsoXky
yDQ/XC4qk5W0fZ+6w6vjydDxyCerXq2cNRfwSz1VTJGq1MVfT85s5F/OA64R
nrW4IR3D+yI/ZjJtFnVGJgR3ma9WK6GYSV3U+4/Gvu5F+2FycYNbGggHs1UU
y4V6fD6yMUu5TAjAtsvHmFZvoThzCw/efvnpXoyfRN9s6MtqOTX6XavjmZ2O
efJ85FEuF2HkCuBG9egWvLWrXVrkG9tf3awzd+bJ/pqfY/Q28BjSL6cy2hOT
+RMd9+qKavzTzfmZQiLxnLddWTa/0AsmJ1f722B63MaGNyzeZ5tkfPuD8y86
BKqb43PAjY7wS59eOvFSdCNOvmVCfHQoZuW+1Qp6qQu9uTE9zjyFEpINcD+5
TskkPEz/6rPPdKaaQYxQ6Rm/r9zdznx6rtugN9fP0PpOD41p1fWT1a+ut1Rx
d5OG+Op4POfpeGT1Ah6btnpyCmajU9I/oP3jW1Wd3Uktm9e05kfzYwmxlgnM
mqkj+Xk7Npv+O1XeyS4ZHe76IZ3daLVQFxGsNr+83jJK4nTDApXPw1HF7lto
VozN3mJ6kWYnmE0AZaKLumHOLhqlf7dc1MzHzCBn1d9YPpfPIierYy5YqHuh
uXh8PLx4KtQXvz5f2UtP9mN7FV7oYBZq0Wi+ju9HrVJZ5+7KxVBaTT13grz5
1bylqs9zvvKaLyw/dMa7Arzawlyd8erh+N7wBA9pMvO78Vhp5GFNGr8278KY
JW0yvxciuFpNUbubOWMz9/Rv+59fXa60wcM0pX5DfTLlXtSaXVzjsrEj2Vyp
pHqKm//d9eTXq2yLJtPFG+Sy8FA0RmwZwlAthv9gT+zfaLK8XmVJuKZ4U5/4
Qi7LyAN1rW6ubTWXZKVKmnanXU/9vXr+Kaj8rFwWfkNtBet6WVK5OtMiJGlq
9ZDx+iNykRiVfB7O5/fUjW857XnSRSQTUty/edaIuftklXn9V+QS5GDrtjbv
lMtFFtOPqpTk4xkpK3NJnSbHfsuN0Pec/zv+xT+InMtPtn15t3z+Jd9oFnWz
KcTZBgGudX+rH9mekBk7vkku25Tjqp0laHVV1Fz76f2h1dsPdn+tfky/c5Kc
Zpq+9WOIqGLr00IflLgJspxfxqabNWgLaCB+ef2YyD5ZB2vMZU2EWrDM5rIU
T6gtiZqIBdQa2pnLAgwlppioFr873q+a3n64/2VafjVTYPOYuEV55JyT12FL
1SZ8pq5qBbcQgPllTZjXq1wNQzV8sn8fGCpMCXuTiXk+8kbuQGxOUHLebLOY
9Jnantdofn09f/apfrHUgzQZn3pMLYpWLKsfnEpVKiozt384FwvYKA4V06li
6xy0353fdxWpS/W5E5bErBlmnrVL9eC0dKIThuphMqb+CXKZ23tCGY1EJLFp
hiyi2S6azn7bOOwN3rK3xSfkkgiUFeOrFgUuZOMgM6jTHOGrycKtGWex9DAm
ZY7drbpLZX61HaM/3p0/148stqDTyi3gqu0CiKmW+8JXxS9RP6BkMaWp4n92
uxvQqOw3ZcbKzaXt3jYfxoyDYRWXrMxVY5cwyxPMVdJcdQWQ3lw5d/P35sNQ
tH/q38bDUNrFJTVKqeRtF63m9Z4qy8xmpsX8yfnJ6gGueVlfKFvMK8mTK0eF
UK1flA9EiY0/dyPvovb5ybY62PfyliJ1Jos1P6pmWGDEw3m94u8oy3ZceZby
k3jsui5SLYNKM6WWu625sVx5FvUmTKcpZKFTf2kg2UY9DLyBKN8ol2khbWtS
iYvRmYBR83aE1RclMS5wvO/CtSUzyvzNuXAlJWBOb/X742pPONbVsyPjroHr
vhTvVwUvzkR4bHwQfxKPZfI4DMf+nX4/fQgMmVW21dl6XZo0Jy1Zx3SdWpQF
gbsbrTC/2+8Ha6dIfDie6AjL03l6xZT3tuudSdWuXtZ7u1ZsH86THAqLzfRh
1Cuz1D/DyE5lz9aZ/Qq59DRgZ9SXmvuRpZfV0ydfrXUApKO+qJaouxWhtcxT
6qmwIGYsuRZgI5I0F2V96tfNxrxe5ZNZDk9wR85gUvtrKgl79nxko4kuRpbY
umn1QJApe2HiluvmR7nMKM2pG4R15qaKc+Hvz5wXr/Qlfw8gXm6+Yia5UGMl
ySTKRfW3+5Ev+lSzqSB8PB0MTsGORf1uo/x1sfEVN7NQeKn0zZKXmT0Q9zvP
/Cvz6Ki99hvo2EYe+VDrw5QXCyXLxBynCLF8ZfjEkgwDhaxHbbDLYlm1tmic
aXEjCTCRnWN72Qzv1DOtLwwVLjs572Dt7yoX9FSW52pqVgnJjp2mhStvO1MV
Jj5YuCW91Tq7IHlEryZKU3TqsvFru4FWjIrzuTULWpifMH2hvFVeKqKn12gb
OzRLv3++VX20oiP16mxDJFuooq+L8b/Wnb4c7HO9PfgbASm0CKvTsTBijoWe
5M7WA4KU+xkYrbz5x5T0YcCQhSacy9K/Hr+YVXAOmCwuVhDLZbdGlPJKdmLF
MnexHmksejFWvTDuwvy8xObtVT7d+P3L5yYYRAa5BsXia3YTgtiXdQVMxkDg
ukLPd0mX2uvWwI3y585tgOkX2mPuWi4z5lm/Qf/+3Tlj5TNSWTkIF/kW00E7
avobx1OQUzdFf2UNxVyTexE99itnf9leeTVVQz8plbsF6kL9/yp2c5XRZ/kR
udB4EXN5qnGa2qdR9yLaqQhWyzpjs9TBe8QS8PnPF6lGhkvNGDR3zGIudCtV
oKnEDWy0oLu6fmoRFnjlu5AGG/xY+iA/wY+5eh2dwNG7uMN9O4XwVJ3ho2qo
LpuOcks9TMSemmnehSLuJSzKY4G6HUsOAzSnQku1dUx99tmjA8334JNp/hiG
Jt2sy35gx9SiRt/MEEi7uWYpzRdhz06eHwyxAe+ifEgc/0QP1PSB39bTzTRZ
Dv970p9k81UrGZKra2j2ljmn5h+J5mp+8uFYHYbq9On65Nlq+zGjResu6hD/
UNuLcadq1g1h1wP98OaWvYmm4UxBJ8vlnVoSX3bZDeh+L5/cNWUhpXWfq+tb
zBO7GAhTj/rixXQyQmCbNNsSNxbJRnjApq4myxg9irt11m/sOn/V+DxKtBnz
bfqSMnGnALJ8UvtVvwWGphH9tVetTjbLu4XfNaEzdSfWdikFomZZcX4xL5P1
xWTZl5yR9Q38/tnF5tfP5sX8Jb1ICUcnHS9xgwSMb2vdbUTtQYziCxebVjTL
gnHvzZUX8OHe3IyLsQ8/LL//mXPfZh5Gt2Mxv0gRu2XShVEAQk0L0g3mcurG
I6lFd2KdGPuRgc08O3fLd4wx/1ynyT4fBgwPIxUzLqxjqATb7/P+P0MudO6b
/VifhZrjOOMuxoVqKuWHIGCi4OLVeMTFCAMW7kVwOUwsZa4BggEbaD4pjifX
HH4vOpcV3RkbMJW5AmRKbU11RrTykLv8nue+lWXZfkxfxJxurnWyISHVHNk6
6J6K+8K6ZjktnpB8msb1vCDRjF1jsdzPa7UuivVhVZGolrPJRBrT+Qv0Rd1v
s7gtF7M1etVMfAl4MtspJ7VXcuoxVsnuJAAY1l4hOpUEqDV7qtZstcCGW7mU
VumLYvIfUhe40cd3sLb6kN9fe9jIijGwCmyAjIX/6CwEJgzxlaQGtY1Ss1fN
r2bRAcBaWI9QQKzXGNJKh2YkxKsW3WhOafUjM/wb/oWCt5flYjYRAKJ14l0E
UyiaUioIVDotuhpWq7e1HutaXDZPSVB1WAtZrEbHdeuzNV0XVjXniooEMTB7
ux4qMz/kuJOteeOUDfuIHdv4c01g8BQPTRrRWK+tSgBrMQ/G152N4AFf9Qve
ngaXxTu5a4zLaeD8U9y/zP+buZBZ3R+68QP7xMsjd1qczVvq+lgQAZ4lia5O
C2eAdpPx6q2h/mfn69jsT/HiPOeXrRwMHt9YT9XNNZcw98R4LlsFl/q6VUgj
xI+1YxrzxnEyT/eGektasJbMjDIS80RS1VfbG8LHjnMxbjEPtsPqRvScdatN
nRKdmmsBVG3GenQbSR0stpon2pjUXEb1UVdHyCSI/P07mMuXSYcX55BMzDJZ
/NABL0/FkrqutaWg0U9FET7hLVCVbqueksjlcdyyHaftL/OJAGed2eLu3UXg
a35iH989fvvp81/ClN71ca+TW+igJBLUWIzVTZZIAKUcu2RJQCrwd62sxeIY
pfhLtQmfkRVzc15/zGuNv7kX0wyH6VhAv/6jzE/N739aLhSkcG0fD7pMoSE1
u8KZeHy3p6KLtlWeB4nZLq5ta80YIFK6mcGa8EEvNaIOJlwl/kel0YQC/RUM
EXFWjfkh7r/8uneYiyFVnAaW4JGDq2Vdu05wGsZZ+DLddjr0cP0Xp7zwsCXN
B9GlzU8tGS7NMPNLe0kZBIo8te+mob3qFhZIh/manykX85Y5V+MpPTUZJ3LS
vkak4qxUcWeTaUO2mDyySgPK2wvKBMVoWPJI8JuurRMoqd1FwZkZs9YjWDC3
+nHMNz92pHz5SLXy9fGWApyjgQSkQtoXa4zd3zoalmTIMaiGQZtuwGy2LD8j
3Xx+hgEsqNEqrsZ2WOti9aahj4gqjbjSi9FJmXt7TM227wfbsYfnJlyHlVNd
N7yDxapisQUBsF7Rab8EljWvqGV3oBvkU0QPxbFcfAGiM254RgwToxlCbYi2
DpHnEdR5uTGT/Gbx0aoM1Kt5Ayn7nUr+3uVfVgnOGI1g4zu7jLvrMaXPhkaM
8NjpXhM2kwx58a1AoQlAtHWr6BDHVRGSEKGvI/GmFmcouU5dFsL4uf95PZrM
mLlsd1G867+T63mj3zdmJp+UpvGHys87t8e+bpPdYEWyUsdcgIaVgzQYYsnO
dIHSY7YpNOuLEtyJTKBbpNAkioBRMkBcKgQU0yQA7kZb11vqrSNP14efmw0l
M/+OfC5fzoM/NX+s7XrIpO6nLgmgJJxSHuZhF0SGEf2iFDIybU0MQM90i2Nu
hQkaElboXKz9U9NKd45ea9mBRKY6DWaa55o9NVyZb1qQoiqbZ9eZn2DHVsFd
+UJyP1KIxLfAKgkcYFmblO9X0BAy70hhhrZraIUNiQ3Mv896F9J+l2MBhl3P
+q3RIbMi8KczlRM4SByAuXfCaDxxUC2TPauBKOYfpjXLZ0cjiFX/y0NcHQHu
uBWttR05dwefTsIIdOqL6cn74NWuBwKG4fOOpEa3ZX1o/Tx3CVDBawqFlF4e
8RPsBXJkBizMxbGGRQeHfuO8DEopX9Q2q+ipTDwZy39vfVE4IHk6PukJZA13
oXtvssgYS+Ava0FXRsxLPd8txIFyFsrvk3LZrK+trWvZasnVTeRkmtj5Wgcq
MgaWC8ROxrwmhESLB34sCSq1PRk2dsLN1tJc1JqPZWnqon3KTJM5Ygo1Ssqr
b+73Tz7WZqkHeWQEDSqljtFM6SnBpQEBvO0kpYpNEbSrvWtt14omJ3glFVCa
I/rFdhaZzDjCV1uj48grISmapMoZkGB1S2Lztm27ICNtTG6KMgGpZokwM3Y5
rKR1y/Pqlgrjxmlb5mJqE0uGqQLjLw67+6Z27DhHB+rRea+cbNepbjJVglHZ
eOcFIpLGSvBirdSih20y+EJqXBlPr65JAlnPI0s61ytmWmDQRBarnoARfBMQ
vngxjpCZ65LHMeauGxPX8yGLfq4aMzNKvNVlQRW4ZgnczHeUC6nJ8XxuHvUj
T1YicDU4beKOBEB+3nZCASbXvZO2RRRjTA8h1FZL8jy0lFhTGSAUS59KFDV1
LVH+kqs/ARm4thy0TQ+7BpaNNztpZWuZtPHQJhN8HC+bTTVrtD9IRJeOwtwK
Qc21nnxfuYDuRWbZRvyiHvbvp7wWQC7ISDgaXlsYHGTz0U7RNj0sTdH6xsF3
oOaykICqQQYQl9ISAkLNH9XFCo8YUwbZtVZH7ivouurYnrYkEFAwoKclxFPD
TKosHQ83zpBH1Q1VmgUZ7jFFfmvCoxH/PBnwdFw5yAd9ZBeiREkeVaM4WfRS
o42iszo0MFuhKGTt8EWPFUdVZouFLUgmfd3qrsFCF3QCH4WhFpKDVWwOFqLU
Eq0A4KBDxAqwbG0QgGroovEtNQbCI6ESKmQcE5FcZc3FBaE3t2GjW1KfE2zz
FNMY9f3jfaCnqC/43Z+dd5FKvXvNU8bAhAlbWGiHRdjoRV20VmK+iaWvgcPa
Xnct+ZVQWKSxMRJLdr1ukAeoiSfTEuyM7GoBOK3rKW/A6TGNVQwtjB1wWus4
/8L+DaUZBJq1XGyXZaUZFZqnMqmIDdyUIiVTGGgC9D9r5S+fskrnY1m9rGEg
wBwqw10AjGpr0doDlslqapqDX+kbqQkaoziml7buEcTArqHEv7IH2VCFBVa5
UR1yzVghxJfSk/6QEes4yqFWJpxb7oKihItE9kBbhr2A3bzadUyR2sU4YDPK
ZXTry/zZKBdlskWTp/gefUk3D74z8znjzw5U0jVsn2ubGsQkuluBAeq+gNWp
G0vCKChc0eBgaill0TR928s+yKZrQw8LFazD501bQ6cKmKjG9T1ynMRCC5sD
BgfOH+O5gY5Ew9Np3gi01BNjSieOd3P3ZZYtzpnTHShNs2x8MVeThyZhhX+j
Mm/PVy7+CNuSCfF9IVHCaZuubjqpsfNFgVgFngeYoMg1ZBS6HDIqGo24BXfI
ntxMjyBH1l1Rex86gnG6w/s0jcPuJQJMKg1ETO6QOmj5pwDQ8RjmVEbI/hCh
J5w4hZ2dFKupP1lqXl//1kqNJc6Js6TAVf0CuYy1KFhgVEYiEWl1G9qiKKyX
kAoOYZSEswpTS3oRXka4pu4aR5YNex55M8hAyrqRxQHO6FCg5wWoznnkm1tn
egNQDcsPkEARZUsoHIwnLCFUj/jSqTt8pGagZz7mlxeRZeLbvNUfGTz4U/P7
tAwC6SsLR9Egkmw6S82TBFlJD/ouQ6tg0eQE1XqyYYWGPiHI0V3f9PBJTSMh
m+KA/20BYRX9oQUoQ5QfAp4ge7idTqMrRhbAfFQ7A/SFzL+o++XmEKnUA5Wc
nM3xqcQJRIO/mv2zzhW9YSLDN5MLhdMhnZQEz4D9LpouNKi5hBsJQdY4XSYg
vSI9FAYuvij6NkcmBjJSQdIFNZFNVeCeqtAYTqPxmN63cPx133eQC0xd2wES
I/Z0EDnkYmsPdOcpK4CsDkwbODFN5gy0f++BDLqu5bxxG1tBtg6MUxvd4sb8
owazd8vFUkkRzIMDBEbUqEgYLVkZcvUai9fQ+kp3aKSsWtmQiykAlgtIkG5t
WtvA7Nm8wOIbabOGvsQ3kQywVVGjMVc20B6l8SQkcUCVYdODJbM9aDhQBi1k
qC1Iz9B3rUdQCvgMridlblDVzFlMW9/sO9wah24u1cN89bjtL7FjiPVgsgyQ
MKgXrFPbBAinKipJq980h6ZtmiJvXVN0Mm8qLHXVw+E0hxy4rMDStmTjij4n
hep7KBW+qnq8UTU5nAke20NrMCitMAh2UHtmMV4B4kACs7OdB20TkG9D9Orx
gfBZKpnlwzTQEzMWPXUTDB5H1M4HCKdvbXadffmhpl9UP8a8H9BV2yKWPNAm
hkGDxYLDASlG6lD0BaIUS1arKpoK//QWHgeyQ/iJu2HM8sbmIPx7yV/hv0Ye
gswPVdMeDvIAfSOvYgETYKV6sn6+lvwzJAWfeLHVPYJWAA+gPsSe4LiVtl4Q
l0pMkWs5280jBIKoJwwQR2yk1k+nN8ybgti/tLPmi3CyA6fSkR1pELArD6Nl
LbAwIFaHNT7kEqoA75I3kFYP1SnoP7zeHCAwKBY0p8/p1QMcjeRvVjlezkOf
Q7vaNofNK6BKHQ5DY4nCQvYBKVHZ9wVkD6VqOg9QcZCIdnAnmdIA2rptgOc6
DlypEgA5U2qK7rlwkErYOTJlsoKqQJGBUX4e3pG4GWOoSNT/PH2BmUI8jwDD
N3XArocidE4eCFwR/CX/ccBqkyKw88D2hz5BbPjugdw9ZKLxD+49HGp7wAsH
EmMu6bkQXl61fWtbyJo0yULBbM/fpedBJ0l9gLQRMwmoB9QIlq/wOMiWeFIH
XwU7yDkIdOY4ENcoNSS1AJAkAwxN4eJqBLr1SinEOMtRxd6AnyYXgzDZIybH
fnWQSiWxjkVNuPcA5w39kH1RHbDU1aGpmzyHLuSkFxWWP8fyA0NjVfOcpFHQ
XfQZPA+/h2QLDeJPyfLliErxCNxOX0H2De7ETTBocEyyhkPDLgCpgP4aUqgu
oISQego8AzZJlhDII7LRfIoTNIhHpHkOjqexXFM+p2vVIkuuwzpd2y9aRz98
Ys3XyEXQrESEDSDgLceIIE5kh2XLsbvJemFNSRikEFAdEsUBPh96gc8PB8ih
4k/gS0hS/AUiUix/zmpT8H301aHB8uNkRzyYbGBD+ghvVhfRNkJWoBpgNHsA
tw4JGohH9xYmUPgWEEKoBuOEPRHXxNAJKZAK7cEfgE3gg5l6FB3wYLsuJZm5
FQGpbT1WhJAShWweemdSq4jTautsm38mF5o6oQOsOUJ4LA8CRtiajkRSYW1t
wVoSLRPLoamwvDmL4UBm6xDlQdLhT6PvwTfIwVRk26pqvC/eVZG5gwkjGfak
R2Tw+IJmNmQgYUJh9oreEutDIKJBGQ4OUkPVugFZ0NBhv4TsDWpzanCoYHGo
4wNZVU8Nm0gGmDCS5Ch3Q7ObH6uu+fCHFvKqnykQMv9SX/BbI6KEfsCnHxrS
kh5ZFwJgBbnxghaQdYD1IKdXWGOihtAHemFaeVYWkh2sFoRD1m0U2oGVCT+F
NINvhMOpGFGQ/cvpm4cmh0hyUiSCAfhUO/DTXQfoZkOLeAksHETHrqsgbhVc
nWsEatxRe9CQSISNW75Glkiknro0GbXm4QK96x6Uddfu39sxyl4A62D/+ZZM
S0MuBctVsfEi2qVg00V7HytM271KWlIVs9rk+SiaKv5Hhi5aucN44XlQl4L8
UxQLpF+RmcxHO3bIwbGR1uArGdWnKWoCeDCxVdHB96Hmw1OcCwABo9uhvyAI
aTg37SRGdALOO6Raiadr+bhThLNu9PtQtdb1Y4vn5Rg0sSjMfSkh/QVyEQgY
kIqse+S+wKhj+RF3JE9Npib6d7ZQrC604tM6j2pSHRYXQ4F8/dr8hqqoZgmS
ucN/VRQf5JxXvCPIP0XHBjb0wEDwQPCPglwAcw8Di7vIHxYkIs11nzIgFwR6
p0VyTtYYNKSJ2QM1jWAIioJctp0ODoy1QLFjJ6RCbBQizEM4X8t/fom+gLNy
gKJAQWThscvBnOSj3yDl4UUkdzEpxbT/k3AWrzMKqObXlmbswNavyg/pjogQ
cpYrKQwZTDBtjLehlJJtHOsOqZTUBb8oGYY3pLgA9RWAOJKzEskGCAsBET4o
xEydaIgpQowK+NZQ2Rs/vVcAAA1ESURBVACaqzyqsfh4IfTzRIKAunA1I2pC
4eGD59B+CR5DyKU7aYCKejb1CV1FM4T4ZFSQPF+oyGFcV97z+WSv8ojOWK8O
l3o1SzNPSGES7Oi9yF7yNyClvuCfWdDzJYTXw91QNCTpt6TfkLA4cDvCXQGI
LQ994ZouWl3yQW2LniiICx7IU1k7qE4klQRwNMpJtfRzFgeRELewYzA3G3Xx
YhPHe+VCE4+QzQWpAcpYgZLPakTaFCdereU/viIwzw+gAUZLyV+TZNjIMTBn
8I40HQKk2hYAmGiYIjKhCEiTgkLApZF9DZmH7ilKBRUuojY4INnA2mHiE7gf
FLkH57ZdjBn57VXS4K1yMTRvF33gtNMK3qEwCRR24+t8glff5WIlBPNTRS4h
YjjS5AjlICHJ+L1oEhFE3ojQd09/EcIAoPEDdVgBUyNzh1wGkg1cmNDzfGAk
4pAoFxgWHGIUq2YiZ65OIQdkUB8k1j0I5bs7/zgaCwLJ+xgCjoFIHgPBbyUX
xhdVcmopoI2gnM0ucHmiFQ72kBPfjVICYpga+CDEPwK2QFBJD0bcN/AvYLZl
iyOeXM5jiQDkKh5mA2I7NZiYWlyEmOkIlKs5HF+V38cWQQhGQcNhDka+mS2b
TNrol6Jc2Igx7E4UECG1SNYhvmkOzPDB+wDHWQ8fWiFZiiCNsrNI1SH8kch3
87QV5Bt8mpg2eR7tblXbmy+Vi5BBQMGRHkGGUYJVRrU3pS3z7yaU5W/DeO0Q
0Vs1ohGKr4BR8BIZO46R8KEuYsqC7BgFohAO5fWgKXBBMG9IrDbg0cnHGJRl
d3F6x7zwwf0jPEYFkBagCziZkBABqZwCiGqlOIfvZdCqplhGT8kZFvzLNyCQ
SD2Yrihqcp0Un7aBqkgKTVQBHCqIN0t4AKUKgmvnxTRumCepUimwep6J+aK8
mI5t2A2HC/THVROb9T2vxDAklo5pHyLwcopFmcojU8UBEKUloCdgMkA0ofyj
N2zdkAJEFg6cNbAXJ97qsTBK3OnyvEqCfg0eU2mwJcjWQLgd5axUg9L2Hf0p
1TeUyqTBZL7yPIY/8CZV8vzUfkvpOqbnDlTFA/PcI42NdB6UpyvQtIamK/T0
YGJUiApiVxtU2I/MPitfHup8ul1vIAg1IourmNQgNTkkUoQgDGWsZH5IRFY1
OtdNnHS9fmQSq5Fs5vg+BogxoMwjG0PEQJ4IGw4rq/F++j4rBT8nP4zvAQrO
qyYGn5LeSGBMsr8B/q1gmmpkSInQbHMiCBqqb3M9ZabBBqBPHWVsCBs78Gic
uqEcQa9VxMPoIOzU/5EXU4/iyqnxCtXcSIPk7EU53iYnEynjuBeZCY4IKL8M
+FLAvxZWUU1UJT+JFznnlYxcQj6nCJaMQaJjOL3DFGlExMyaMluTo6ADKZyW
cjcEtXoQMuj1BB9uGxTxgIZGKIn8WocyNqqMlp2DTxFoHuCSDB8RMPenR5IZ
f3o6xF5k6oMFGk/VjZvl0JvyuRPEEyFDo0Y4p0G5EyxphDtxw6fFr9b2bdzt
a+XJk6gq/pSkQG+n1Z7p5hHucoZmJNlY8PxOjqYSi8khCBEzB3LolK5Engjl
h+BcKurDCchsCkSPqHbnSitHqWik+iTWHbkXqnUyaJ6iyRysEYH7//x6QJD6
RCnTk6dSief8SzxuB6XCLdEvBaN+Mswx687kEk62wibsY5a4YgHlMf0SVWlS
lHxl03KW6ZgOKBKTRppIOpDoabJ20azF0JDCD4a14E9l1fDvBJyLks4WBR1g
VwlFoQMHDVO4S1NvIYo4FLoza+rF6lAy4rpxPwLxx5jcpeXwcSaDms8Ishel
ZerjHWflU7ZLnqnPwozn8d3PiqHnFGAEFUpV3NwHTjGS7YEhZ4uUTxlJIj8o
C8+bmvLCaadfQQTGrGSNqnHti/ggeixlMalaJqYPqDSt4ioAwhpEaIPMBw8G
2eCX8ijGQF8g6gyRWaFN4uM5KKl+jEzTNJ17WnazdVoZpJR8urpKiKnFRADl
zRf5F39u/bm+/Q63Hhp5tTuoWaymTkvPOUvSkIppmZQkGS3VLJRRa2btqSaM
wPVKEAvkQOrSUJ0Akz6IwC0VqUExqYIJlANsDbL5QXHjBaphEWxjHgMxEYKm
OqXze1Od2HVV5Tw4/fWzLmYl+dDUpvIJbclqOry6mPqRL4RCZ+VGpwfKlIqz
YahodGmnQYmjEBJZdDIYqCTi9BJKU2HGWm7YR8KpprYkWDhcli6qriS7E0ld
Tgczj1ghnkPwQAErsvNkJFvkp1AVS+E2MtUca1OGVNpOLE6EUb1FL4AeQYlo
heMOTP8YtE7+QbzqKNTqbvFVeAzHKKieG8aaq37kwMMJo1xQct9qLKXlEiL2
HgVrRIwHYhYkUoUxpzJlTKiUIo/qwjApeocxRUaC6T1E1WryEQqTx9AjQD1K
qKWgPhj0lIcGdgUwyI99r6NrxBCBOnY1k3FBMB5Py7Dq8aw1sxz1vxhV8pxd
evrk08/IJZPDhyPR/foifekQS1aN2eXyzeQizq07t7u+fCu5kNdqzqdqNJS7
XL6DXNQsHPWYh9mvfzR3dNeXb9nHZ3Z9+V52bMFI7nL54XPg9+tzcnmNVRNH
DItZXif8f/HS5vVb7ovX8Zze8kX3HV8eUb/F9qjhSd7ol9yHsV/qq+/7bAu6
enIo2W+5L3uUTf/S++78+luE3/nJv/sX3JfNWSj1/9/34nEK8q/cN17yq+97
z/QT88fu+/If/BPPQv0+l/pH9z0j5RNjGHksT/KelPH9aLyrU3mvdXe8rz2e
ynvHNIvhdOyeeN5434PndedTeWweP2+679Hv1xzLgacFD2V5NxNdYJa74xU8
neRbhMJSyIczyUdWD24+hYiw+8G3d8DjdF97fjSfNk4Pe/C86b5Hz0O+/6Qf
/37TfQ+eF45K0I/Omsr199B3wCz3oecb36IvKceKucpRX6oHiexjOqGA6gTu
BU3jffWQiTv658p4XtKj5033PXge3XF+4nnTfQ+ep4/K8HJjjq473s05CsW5
raZ653lNQ1sPo9reVVeMLC9IcgOqIs7d4/t6PPCOPdHH6jxQOeOD5033PXge
/Slsxh79fuN9j553OB55YU5Q1qO/41GqdKN8tIJPqwtZkQrJf07OCCXP99QQ
J1YO1Ob+6O9O99GhR7a8/cD22PlD/vh5432PnkfD7sNTcon3PXpeGIIZyOE+
kAsMTvAkaTTzGTlk78EiqjiejsdU8ueO2WOX8ISdiPfR44/mDk0HFaAjHB48
b7rvwfMyuuspO5bue/A8su5smR7YMUM3FlUaKXPM3qEwXB9r6oEqp50x9h7T
jNIyz/rywK9O92GbtaW7w2R1rqge++npvgfPi7tBPPb7430PnteeXSy5e+T3
7VkbvhHTbG35JpTMB4YPGbpygS7van8oT6c8Q+XyAxw63decjmd71/ueBtE0
D3HyeN+j5wkYG8xSeIy7032PnneAuzA0n2E43cfJ+HlDVvX493TWWfZzQre3
Tl8z6+OsH8QBe9T9IK9g3kd/m/839P7tknm30rz7zp2C+qxI1C6QN63k5Ygo
8X9oi8r+/emi314yb97LTxfk74K5t4Q+e6PCmBfevTv/uyu5DoONPB7DJ8Si
hv7J4EzbffHv7djjOrmqQ5LLOtd6haXNdRDMn1X2js6ZBTRvh69Ag79KX6oj
TtAU1Wko6TjfIxbrMJRjHqkth5SEAo9SnqpBIMrG/c0wUB+CHBDyZ8NARB7J
bMCn1g0g1xNFgdwZHtWUdPORchZiqM54ZTiWTTsc79PLf/g6gkXK9CmAutal
TEmaqhSCCEaAJnvUAsvMsTxoEmTqwGvosy/KIE46gD7PD+48nq2bDWjRPwrk
oXTio6qK+vDLkJ17kgv4rGFwaCqVgzEHHNvodhHckEsliZmlj2c6rxEksTlg
MclT0MGy50RCEx2miMs9IR197IirLXV7PJ9OVTgPNpmkASxdGdxJVU18oQwT
A2zwaJgvYt6PSBpT609R7xJ4Qi7ktElP8pxJdNKRbjB8B6cRDe33SBVSEnWw
fcwsOjuk1C+JA7Zp6E/uSi6kL+CfIWUFLR3otJS8lLsINgHU0Ux2rDtBFI6I
1+okXPL/lGtPchEwcQP4WWQ0apbLuQunjlrZMaM25UsGpKJg7rrTyL8v7Jg5
a4f3kyJGfQnTo/cruxzr4JLfhwM/dqopj6dBoeDjKOck1Lh4sjxjXQVAwWCi
vpDfP9kwHIcqiQHv7Mka1uMw8IXfb05n3DdYNpZl2RzK46B3GWxC5TDHk5rA
kRkTixshPPrzq3yBjBeTONQ0SJL/9Sf3VIJpjy4fXdjXpRwHtuTFxeFr8R/k
qyr/iD8xfONunt7oa1ZcjNEn4K4hHn0IY1Se7L0wUBQoyjsXbpKjQLRzKg9+
X9nPU/7m6hQCszXT5XnqcucmPy+VCxGMjK96TBgvjjbcBfFuuTy3rsr8y8zb
n07CmMcjBvc05P8ol+dKVDYUxtyr3til9S6dUTfO6zArg3dbuj9Fj/4DqTgf
Nvv1qI0AAAAASUVORK5CYII=
"" alt="Scatter-genesxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-genes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 4</strong>:</span> Scatter - mito x genes (Raw)</figcaption></figure>
</li>
<li>In <code style="color: inherit">Scatter - mito x genes</code> you can see how cells with <code style="color: inherit">log1p_n_genes_by_counts</code> up to around, perhaps, <code style="color: inherit">5.7</code> (around 300 genes) often have high <code style="color: inherit">pct_counts_mito</code>.
<ul>
<li>You can plot this as just <code style="color: inherit">n_counts</code> and see this same trend at around 300 genes, but with this data the log format is clearer so that’s how we’re presenting it.</li>
<li>You could also use the violin plots to come up with the threshold, and thus also take batch into account. It’s good to look at the violins as well, because you don’t want to accidentally cut out an entire sample (i.e. N703 and N707).</li>
<li>Some bioinformaticians would recommend filtering each sample individually, but this is difficult in larger scale and in this case (you’re welcome to give it a go! You’d have to filter separately and then concatenate), it won’t make a notable difference in the final interpretation.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-umis"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: UMIs</div>
<p>What threshold should you set for <code style="color: inherit">log1p_total_counts</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-4"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-4" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>As before, any plot with <code style="color: inherit">log1p_total_counts</code> will do! Again, we’ll use a scatterplot here, but you can use a violin plot if you wish!
<figure id="figure-5" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAS1BMVEX////X
19f19fXh4eH+/v6EhIT5+fl/f3/8/PwyMjLv7++amprp6emKioq8vLzFxcXM
zMyzs7Orq6uRkZGjo6N1dXVYWFgWFhY+Pj5jyzDVAAAACXBIWXMAAA7zAAAO
8wEcU5k6AAAgAElEQVR42u1dB2LjOAwUiyiKpHqx///SG5BUs+W4JZey0t3u
Ji4phNAGAyBJnrhkwpPj+uJLJfrZt+jj1P6P6+ljluo4tK9Xl+mv595yXF99
PXv3y8O5/EAXcyjL/yUUjfjqmVhMHGf2P1z8WQ0w+XH9D9eJ70dkOjwq45/V
ld9VKSmP+/3dK79pr3C+/u+NX5GHXL5XLl5ntIQ8ZPKcvshDLl8pF1IVHS0a
1/S3DJ/kx6F9p1y0t2KT/VJ6RgYOuXyvXGK4RqJRwao96Pe9LTtO9uvkIhPt
zqdzlmSnPG8nUd2Xi9T6gDa/UC46UaJKxLksihW+md/1+lqrAxb4Irno4GIg
muRcZsUSK/PzA+UZ5o6T/ao4Ofh9kzS5SE/5KOhTZfPxlNyvGhiSaTJFdMf1
ifoSIjCZsLFPnEmGMQZm9/RFz8no4WS+LK/EnT+m/mNxmpQh/xh99n6J3qmc
O7zMF+EwiRoHfIIDLiZ90Xf1Jf5/yOWr9AXawk4Ikfsxz0fmz/x+vr9Aaocd
+xq/r2MIJpdSgJzeoT/MKaWcXnCI5gv0RUZEbBKLWuRyR2H0RnGO67Pt2Oe8
47gOuRxyOa5DLodcDrkccjmuQy6HXI7rkMshl0Muh1yO65DLIZfjOuRyyOW4
DrkccvkyuchYbjsq/z9FLqH7QlLR/2Av/xy5gB0rL4kZx/Wj/MshlR8nF635
0rd5XD/Ev0jw/Bk/3MvP8i+hAUOog1b28+xYCMcOubzdx+ebk7aP5cc4km/v
R5Y7QdSR738zn3+Whd4o0CGX75bLIgy1mpyYH+HU9/p9+Gjlzvl5SNx4Guup
b+J8pB/fKxeCFx3jYqyGTg/p4V9+it+Pcy7yKmeJyGOmfr8f+bi+uv/F/8Vy
31sZ2pDv9yMf1/+QV0JBxLlNZrmoF+wYmpk/1Mhkb2jTIZc7+mLGDC8iO+Yf
eHqeklLCJDujGnXoKpNxOlB5SOKp/n1O/ch86JIiS/Rrfp9/NEDTd/yRVh6S
eLIfGaMWK3b2cTId8fNy0ftwGFevDzw/5o+ZKbfE6fFX9EXfgCm5cKFpOcwH
OvzLE/riR/jy6X6WyYPzLR8PB0Il5lCXJ+SiYlVxwcbkG3nlnREL5pDNE3MV
tgZIvpjvy/teR9ljLNbX1sXknarYHB9vBKTMMfL/K+Xy0EAY/6LVK0krudp9
2b9MBPg0uWyLNLvaIklVYiV5m8EcK4C+Ul/4Y6eo5Bp3kVrLXYHofxqc+Sy5
IH4zTNyWjM9QpjmxctEDGcaay935mv/wpKz/S18gMzmHBk5fgzE7wfO/nNp8
llziaN6bB2mW2diK1+ZuGKc5GID/sCXLP7OS9kEAJS9VYYX37F/in6ZD5bu7
KdX39SXJg1q+e8r86Bf7iXLRIL+c89Edcvlp/mVMa5aO3yUXffTI3NCX/M7h
/w9ykYdcrv3LGbX2Ov8+O6YOqeyecnXGLOv6++Ixxw+57J2y0EZx8W1yoR0l
h1yuT1nm6hn/EtPCGcmSkdgcFpbKO65EuVfLBf+aXFh7atvWPuNfZCipTOcp
9YxGqjv+HfTnXYDzWCR/dcoNqPvjmDUP64vSq2HxgXMWBAJBMX0Hl9F8X2CH
xlwrRnnnXO50/pkFIuZMvULG4EdV+fqUOzWM9N/j/sXv4DFTA5OpnDdhHsO8
c8ByJyJWkjPGD0N2ZceS1lrb9o++IzDAhI+rJXdVYuTEdrnn9G9GZIe+7Fml
ewY+33uDmpgtcrJP8q6b2K0ghzcfIdlOXjnS/vdn4rEp8eEKRmyKAbTUh/v+
TLmcSiZ24tdb75ATAZD2icVKsprURd21YtET6dvs8afGyei/i4+NYrMfWd2X
pJzL+/qK03LNz5CXRsoPjmn0TRhZ6YcmafxtfVGJHSwyS+/G0flyyrVfj1yq
1Tv0rZtaLvQLb860fuC+DgN9kn0O+eN7yuRfr7+ckVhOcXJT4gVDcSd/kYSn
BZYrN5qW8Mo5lblof9FyqxhhjF8SKH+72vJYGKAD7PB37ZgZzTrKcnhBUawX
8p0v7kzpM8iIvYCPpAT3R0k95lozZq7OdZ3lR7kk/NaZ6sdpAVr+aX1helrz
jouh33U4nb3TgR/vr/uR9cIK0BNjL6Jf04diTUzS2jGjt0v/Pjh/gDrmkV9E
8T+F3+zEY7RKNDgReFzSF6NMN+q5bnYRCsjlZp/Iq7FDY7ZMFzDY5qB1cE2u
VhSU7aE1gj2QaGoh+N+Wi2ACl/cOBD2GF9A/8hazybDQciwX7rieDSEQf7E5
ML0JoOHxZVQuecOO6Yfyf87lX2oAyHf8rI79enSY7OwSlqhu5FODan5dRUH/
io4Zi5arGEpHQd84b1I0XfJN+76MoR2v1FXU/S9RyfO9CEhPfccY2HM6p3E9
strtr/TlEn15ckFfmPvIUUtJvf3XPXzaV/jFhdPh/xZNZrfK5XOPfAmT9MzI
38n39RLPemVbzhLHXquPAijlZ2GotR5IPTePbxplqJbzL3WU7/EtgyQu8kd9
A08OQ/2CT1G12AyUI7LArVgrppFaqiS56IfxZQKS0Eo91D8ul/n+VivOZeyz
k3v1lykp9EIxU+f/yvd/EAFPWNqluVpCB/k4P/cfqFe2FtadXeURN+svITnk
fK1Yi1SZu1uLWaxd6RbUcwdg0/+uHRt5dZ5xmB18Kr/TUk7j5HC0ZaQ6ceNc
xT/K0/X6+E2Myci+bc2o/qf4ftennGubbR/WH71DX+DwlGYKoVfpEE/cR3Xk
C6WQSk4FZn30Ja1xmA5jrZ7mwRIMNiNdRs1LK+Re/iFn66Uv2i23ipQkh1wW
C1/YhBXPysXxKS1cdGUSgzIL3hbjaLPBaXYloI965eaydMLFC/3Isd1sRU5S
zAcPjE23/9Qyaar57M1+f5qc+IKHXKKfzudmi9f4yfyqQ9lLx89mEj5r1T5a
01Oj8o0+2mOuwvrqwbc8j+fzm7xxPY3JSrDAwgQX4uWyUC2937/JzdC0kOTo
E1/Cp35sqrLSr05cDoqhHMQR8DRmVm58wijnrrCbAZcSSaIPuSSbJm7zbt+r
DhOS1GLWFrALwE2Qn4tPGLHz/SqWLJWDQy44x/JE1wsT/WegbFICVbIrnD76
cho1WgdwUlXM7Hkp/U93We7UK9mT7yA3XtKf4BFWIOQG59fBLumZe+GHKc4Y
jIzAi1wVYlxFfTTyX/T/13WxkT/Jg124Zv4vfk0b0osehTgrksrhhjb4p16m
9fiYAGGbn1miDn1Bvk/8seZpBFpHyYhmOUVRt0vft5pf6U9fMZ7s5ft67k4K
H7l/cwDGtVyGccUfe5xp7m9veTl8h+v1YNHo8ANffPH2PD6aTE9HgyfnSb6H
Hbt/BPkO/z5GvyA2I5GvxPprOBRvPKCs+IzNyMmqhTaAmk0VypV/SaZhmOqf
RMqu5OKe1xe+sMjk7OlF+Mg0tfPpvWI1v2oTo5SmDIsVgpnTNTPbu0PLIx7z
V9M23eNyoZnkldFLUklhGHkn5wJvjM9FNb7iVU5lHUCafJ7OB/HpWOPfFMbU
YceiQTGP4zAbsCte9ez317EZBVYslMikYREBaOeSJkLiVcgQqpY+tpbysGPx
qNnruKWaqjFXV2mQafJIwnPraXwRHvCq42vS8ugXu3IWlO3n7eubfBSToUF8
ZsPKy4wGTD9+bHx/Ul/4nTwuvzvgpY50pRBHeFsmNmg+dxeY2ZaLc3RX7p5y
M3w0VuE+bulT9KU4aVaS2DiiMOF6Lte41RNaHnK57kuylrYkvYMnR74ZuDBe
EIamiK4RTc7kBLrMZP3QMyuXySaHXDYXiOKJeW/+mOhVLOw7RcTmSqmahfqy
8LOTNZtZr0LpFYzDk2Ne341+sdyFYvI7+mJmO8Zrn7lIPzYGxqoRF7wzNRkw
rkuzqgQccrm4egzuObVv+H0t1o1g5gbPVkZLBv5sUJiGm4ipqUNf9k5ZNO18
U48nvICWJLu5GpxfYmjyallcMpFfkJlQxqIXPlJo1qdh/tUqEKhXAhPukfE+
8rIHSd7cMLe+KeTvlUtLhmfSl6qifmTrQMGc58btEvC0JN6YNougXOU2+sKr
eiXFch0k86VTT8dGtSv9ul0ifWwn0C+r4uzxYNcPm5OvYFL7azzl817rPrCt
viaobEbJeFW5hVrmW8/qpXPYbA5TJCt2k95M7xc3qM17q3ySj/b68etVTb9M
Lkk0VnNj5UlO65FVe9mPrGarFu55+ttXIbUn5vNljN96UyWPeabyMKaO3Utq
MmRsy6/dDfkcf6o6oZn67XXktGLZjCfT/l1IQp9mMOCyX4xvCN4kFxPPjEvW
yH3+hAkYjCtnpdAryazkwm+UKx2/WHQFVdG3B9L8frkkLjufhxmMEdGO5bPJ
yrflEQeu2RwmeFySmr6pzM8qESIANSnNapj4ZuiL80mMS94YosiZ0HdooL/b
jk3VET6ELe9oG8vg9xcA4HxxcsSN4FSknBBhQ/+4ihJJvgqD9OZbgMLkFgdP
vXycGfHwFt/kap2SMR8uWPr9fn86Dt8njn7k3IpznrvtXlF5yRlfTbBgm1Zi
TZbM+FrxShvkmsXkq181LCBZNVFtJOH0YzzlhwLgXx0nL8/oBysDSzmLY0Mo
M6W52MADTHkzVKeaX9Bsu8LkRAMI7THKv3SqW8pnwGb//ebvuTtwQbEf7HPy
W6DjTYnlV23iaqkQOK0rgC3B4iMubqbIa3MyC8x/FQWLim8GMyyN6nLesCjE
A/d9mNLI9S8dpHlTX9R4Q+vz/dVhMTuEMOAonDdMpeNOMBJPezW7jQnwMJTY
MCuEWOZj+wKAlu0kUbXeUCNM+dhvh/kBH49A07+oL8mDZPHfR97B4vwXIoJz
FMWMUJQaQnVw9wte13o7UdGDyVTQ50ZoXsVKsjd3k30jbhmKnqVppW8J9LLx
YzXkarTZnUYPkghXN0s5Uv7kCtz+/DF9vv2r57t2zwWw0dR+BE/MNOHLmzDg
2kSSUsKUIrn4uK1GO4xkYUgGzQOUzjBNhYHEDwKqJkb5MmB24ck82oTzQYXt
RxPTr+YnZ6chG4f8GT4MiaZya34FRc3QlQSoZU9CEGZKFZkHNKOWBOyEtXUZ
9YmqZIztEde9s1CPWx61nvf7C+csXQqg7vLO2sY9M9faXIWsdeUzcl7TBlgK
snS1SmIwrGdNbRJ9beYglq+OXyUbSjPtk30mJJO78ObWvfwi/+LWm1v1I5av
8Yh+xZZ7lYBIcs11g/Z9T0R2yzmZsqrUPJ5pCgqmEFqsURgV/InWqwEM+qG+
Cz33QMn9Dcw/fBjmTh2ZGsbOT8xPDsarXvW8ckuiSPEgTJadoMcFheGRRsY2
96woJfXI6np7fDzMz5Rr6r9M/voejH08+al+ZE3D3TgSwKk9rK58OxiHZiCG
Zj71nLomes8gr4Vv36tpnn/M6Mty8TrT1RrT+/YXH8Jtdv/sGC2u/zbvoqEt
MI++Q8UamBIgjSGWak2InA3Afm1dGzEyI1fbdfF6IKFlJTWMHXXI+oCjik8u
SbjAagu5nZCtbzlxmuYotJJ/eN4F0LC8fCYeYx6DxL2veqMbp6nZgtaEUT/M
wnWlHqUli/FVM+wx4wvXT88hWZREtR6ZrTDIR+mPZvpy+bf5lmi0GJJn9osp
zw/n4Q+r6prci6ZZWUlPYTHXsT48TbpcwWAIqgP/QkYuYT3DDCrqlppGLM2d
uSJh++iR1n9532vTZOfnJpUSMtb62LoUdcnQBugaHg4SqQvNFPNiCA6Ir8sv
M30cJo1ta2LIRREf6NrRSD/IP47AMgRo1r8eyH+J1+eem3fhDQuZJDRNtkz3
guJiZgrgi1g9Qjkl84yXuuTJVW64VOotX+Omra8PzEEcmJlec3hlbueR8i/r
C3wtG5+spPl5IxVDPKuCa7JCtBO7Jd7cUKiaCTxtAilWVzOwFo66Z7HpQnBA
Mm6FlXhD6JLkQ2hM/+k4med+rkJ+1s8h0PDzgkrKdeLBXkU8NEpUrJhueGBf
PXBMPC7WOgO/UvrMNIld0BisDW1rvR28Kg7Ly+0LRl8OltN/U18Y5o3jZOoH
/X7I/RozlYPrxkwDwlVZtU0YRmpCMYDNuD6W+MgJX9AREQscJ7JTLHp2XuuL
YkI5SypizPV20LL+I50a+dMT2fK9O5gv3S51szRc+Pb9gnqUS09aLn06Q00X
zqntNxOTY2Gow5iSVUoucD1JSq3GX20LW1fze//APPhbcrldVt95h2Eh+EIZ
qrbUYUxkVobucOU5/QT+ozew8RlJLVhVsX471ZSmxJhQGlCetrTa08RmnQmZ
jZq7YpF2Kio2mz/YyZQ/va7rCk9204n5AnLDyGaRXIjDBFsmyla1BAlUrAbx
mYSl2Ar9IhYMr+NdXoUBwWoz5MqJsFKG6d2yltqg33+kpSl//x1wBmvrwuDp
ERqnpV9UIhAnN50xru1rj7CwkKtMhLLeD16UzPv74IPqZM1XAszmRNVG+MbJ
NQk9CcN8NxNk5N/on8mf3m6X785tlQsDxrVCU++rp8nAr/SuLxmrQgAQ1MCV
eiYya25Qz0wwfyHwzGEFV4SaZLX8ZS4lKDn39OsddvlfADDz++14j7yDWsCB
UTYwNTD6zN/efnkccBleMVc72/iZsVb0aIxFdCvnXQyuEhAEspYqocINMhxj
96OquLDM+BhM7oL7fnyzOezYBHZilxhNGC/BUmoRLAMntjzO7FOubZwC5s/g
CMBbAjPDlL68TPQx5JRGOF8nM76/wyBqjnoR0hp+PXhLr5mai/hC6qKMmcNE
turgoBrZxVqHPy+XpLJuWvqa1D1xxxUyQwIqS9GWAXVmNWWcJSxa6ZwUgZfp
WZcsFsqwealtjKdmsvkAQ3YjI9aC2o1advuB1r8eeL51+nFAg9K/MHZ+WC7y
IxwGLUc4LQDtGFzR1gnrUfrCrdwQTawFLIacBFtkAe0bBGVla6zRrmw6H4tR
s79nN8c+s7ATyzWXVFggY40Lk2bkwlcyasOYmfcCeX7OPLhBLaVkrX9F2vkJ
ciFXQrkj3foVrA7reGIrqjZS0NswCtgQLSMPYbw0jU6mZIdRfQzOXCyrd0hM
lO4gEZ0PstoOKpvmy4Zd8HqlJ+t6JnNbfq2OluxP6ctcjuI7dTHpwXca30rr
SaS3aaWCBtTeRInoWBjrAfgL2nUhS2+dGt+nXDmAnas2DHDOxP4YjOBoTLVa
ZxJZGVputs0tGwBh6eSiS9Hg/Yo98g9N3Z2mVMkb+198XKWQfPcFJZeGhVpM
Qixy3z1RGVVVYCMxgpzRUUEZZGAiNW2YpVRRjcDbNapjliJZdypFuIXFPjQ/
w6nmYbvs3q5YRX5thV9iY6acgzdOWar8E/qisNGSNovKuN/qoqkROQckYIaW
ELEyqaid2YQ/gLo67ylahGHOWdfZGsxY6/PFVjTGSEWoC4gVCvWV0hrAazjo
uUU2AKITEBpSHBXNmVZTAdQri1mmBHGz3dusYn+aL8MZo/+K30f7C5/GH+VX
bRbw1VUGf0BpS91BOiWS/KKBj0AVxTLb0dpPxlwDX2MqB6aM5zHhnkceTxk7
QoXEFm1bAq5kpQ9v6fm+n3hPbThxv8dJXu7r0csWze1akx3KzCVa+svt2GlB
Rvj5Wl+QFgqfRPaF4VqapO1QOUbZsrOJTRGNVRpupKsrPNvxzrt0vze5ihlI
D4PXsRrsf+cnMdbR5MhpZnnprZKPpquLMG0+dqN2t7/L9cqnas0jl79aLnTy
p/PZp+D8ej8y/bo9zj/RLVB8S2dedAiXjWJdr0RrWe+oosPqqmp5TXg9sz6N
gVaUSEhLNMHoFo0ABq9Somkc1KqKzNgSkiKWswl9m3SYIpYFyunen3ka17FC
pFqGyEBdb56Xv1ouNMKFjZ6apxTfK/1Xha2Bp/Qg6FdIKlGWTBzW+9RFA3vW
1LZzbUUIZtn6chmZqqasGmHxH6pieMIiaC4bhAwoD/QlUV6QxxgBWVKZDDfE
hIQhVSKWHyxeuywx1cuQjI205MKcNeE3+S0Vgfyh1AW/2GjV5F+ufzXH2qIk
W9Sh+NWloF4WTPUWKX6lWwuCTNYygCqZ7XHGJJHaMnh9JoqmTvoGJNqGmi8a
P2sDNwBiNFRxXFtrUWNgCRqbG2DQyo+WRURHxA4qUy/2CZCYx9qknLs25Wq6
fOL5oH4iioxDTPjPVpj8kXwSNt+NbVzqcr7Rhd13wB1rRF+uL3pe9zVnbQlV
QNk/LUqydBBFWdZoRWqrouG2hxdvK1uVFCn7GBiPUE0/qZ13W+AI4iv2jato
5YXzNQKNHbGM5JNMVRfGN0RybPCt1NUaejbRNaa9joa+hvrF+kI/e30+5dlH
7zBdq5SxXQl6XsooKEOuYhmviw46UdWdVRKfdylAZ8GK1tUN7FvqNDMchTIw
/1gHF5SUvQ8HGiF8RFD70AyggHE9tMYQp5xxPzl2HvSnWXLZQma2xWSfW81z
sqeZPyb5ZX18u45/lUvne78PsykOru8A0qsmq0lHXGIH0GFs4fqs0iwDXula
m2Fak+27OmNlUzcQVdX2tkVA3bGm51CnlpxHh9SPtW1boUAAK6YbMk9gCDrX
wJwylqjtXGGquVTC/5SxcVYuwxj1enSmx8zkPOJE/WE8OWR3bZfCqZcp5FPb
Al4kK2rWpJXosx5QTMkG2CPOmsZWbYEnGwud4sbCzVhLiSb+6anOTON/bAWY
jdGMP9G2PRIh0J0bT1KjnJD3VPmyoU8gdPwn7aorWpK795eU27aMMGdrkov+
1fHYA4xmXsATS1tgXqkqII+ktLaqbFrUpi9q0RStshaAS+qaGmlmY9POFrbt
eiiQLeuyKcD3Q9WEg9xMpHPUB9oS5AwRGtd0VWE+fB2KylTP1yRHX0OeoC6E
EHFnSSXLjxfKXSSW+mYSuu7ClL9OLo6q8s6mqcB9nCLc5YUFcbJJ01ZYOP+6
wQdl1lW2KEthIZeCJNF2BURleNb3TdcVbdOXtq7KtkCgi2itqduMwrYErsZR
WZkQgMnTg+dUBJKM6gGwKQTJjmPGKRAWMAXw7ITdaFFdZTb6cm22vFE7l6sF
w/L36QsQFpoxwoohI4yr6MC36OHZVTnA+0M7qtKmPR63YE+0bQFx0MCmLsvw
UQleWZq1tquNY42AhJBnImooDRIXGDKBck3HNMRJ5o1W+tLsf9ChSFw0MYgO
DZBcM7HX2NUR6uRpxs+W2an0r5QLHVORoiJpuqwoCSDDGMaSpSnSyqFjeBRi
6IeiblNEY1nXwNpBr+Ds0V7b9LbMbF0UHXQGSJrtRA/1QsrTC8iFLpIEK6oe
WlZZGD6IkkNzVIN4IVA2EZ/P3c7Veoem2oyWW804U7cls+qDkr++jowxiwXi
JIOjB0ZZQxlY0xYZ6pE2axx0pGVFOlgLEUB4fZd1fUr/QDrwQhBNCnNmG9bD
E3VAnIFhVhKZaMNS2yB2hihaJJaY/gPyADQlASJQs9aGApvQYSmGEhO3nOho
ZmoccCxgOobvjCDRl01Oetmbon0BQelvIaS/K5fpBy7TDJJBSJx14BZ3A/LL
BipgjC2Qv0BPeptltsqgEmnRtVARSx4GzqUYKkioKCACSMSmA6TXF4gb4Pkl
vA/cUEMP9cDZuh7eDOkpqmf0MKNhjaIELk0ANQcuysJJt34HAA0lc9HFAOFx
m4JMFBtsYsIvSLSBPRprddiN/h1M9E/RF0uwFyuyAeleWUBC3GUZsGVjB8ws
baAqDMdPegENSQsEbhAKlCTNujTFJ3gH1KSDHqVt2WdQGlg3wGdl1kJ8fdMX
gsNNdQ2wT4QUwBPwNaBGDaK6BoJpKairHSPgh3o9OE03cz4pNdRai+qlmPNO
40WnUBxSvruAX2HQUyeh76Yy5lvwmk+RC3xIQZVG+P4CuEuaFUhgKhx7A0OG
SK0csrQvIIKWpILYrcPDBUkG6jXgoxTmrYVA8W/a4ikLFnODEKIr4IrSDqau
S4F+Ih2qIVDYMPgwjjpOlwpmEV0LBAtp5TpQ10wFDgGRb8TcPVuKBhJqYm96
A3gbaBynHxgd0BfDrsQ02g9WTE1tt7+Xp9QMcCf4bRocNewZ0svBGlYUAwIn
CKCDfyHHgr9wrhAVRNMVXizk8SEx/z+egIz8Ra/BazsSH96GhAj61LeQVQl3
hc+yFm4ICtdCg9AqqFugn1VTFgb0KCJwoLMTwV5TIR9F8rp0qbnGGd5Us/cX
1doox3kpcYidB7N3pt/6t1azqs3jOT9xE9pnyIUyuqqGsmQ9TRXJhqylUABW
i5e2GFIGrwOB4LDh71OSRuZlEwSUFtMH2fxv/GOhZAU5pt70kAtJA76pA1aA
TzJYvrIWeKJpiZPO6gb/wxLasmz8wHn0eCLggEoAHMUP6Ic0Ef7Z+nGnBocO
+m0nPDeKGqbn7gSFOSosTCCaukhmec3xnJhWPkjhtkmRVj9CLrqCRSqDh4GL
4bwlA4XAiQwVymIDogDYoWzoesgunnrmjZj/QxpSZLPAgtC89ylafMGUlKyD
4pAuZfBGBTC1pkGcUCE5SjskknAwDXKhDEBb1deoR/dlaZFCNc6VTDIwCxGL
VB00CUtpkV9ZfzMxmDEhPGmW6kVxazA1tikqFrBmmugoSqX9zsCAN/DLuFrr
q5jb/Yw4uSWFINHg8CASh7gM59rrHgdJTn+AgBpSqC7IxZupNEuzSUWKbHuR
APBsEawbKR3JxcsLQA/i8R7ysWQDM4AFiMl7hAGIqXlfNRUKOtCtugXsICA3
eJwU8QOMH8LyhlqiqOjdVH7MaQOWjW9oA7Ra07BB3P7IYz13Y2LZAGeAXHlk
revd9HM1Z4VaT6qfoC/+B8LJQgTC5zI4VVD7UvicVFj4dWT+ZMUoEIvakF3J
YecRqNikWRBKF/wPeaNugN6RrAAlpLBu+AapbeGybAPgDW6s6Bsf5ZW8saYZ
EJaVtmINACKA0sAOYKAagE5pOOgAAA69SURBVBEI+VBdaJAYORdmaxP9pkYR
jYsw4LYlDmkLqcF2BTeiS7dRiFU1YT00/QfIpSrSlkbB6S4dKCRGMkNCAohZ
paQibYNwGQ8Uu+e/e6WTYCZHRH8PmQ+s8T3CY4jV4K8G758sHFeHylxPjgyl
NA7KASmaAHBdtALCAwSH/cKwd+hfK1kBVIjBt6D7vG3RvcZg9ggtYJZMG2YM
lzViMcQONDGlCiQP4Vs7MR7S/+N8MzUqq5E+SjlcIz+tj/Az7Bjr6eQspohW
LQ5jGIAul0MxkMuHviAUzmobPMYkmeXvOQ7YSsN/GCSSDVk6/1eQLQxPIpYb
Cm/YEAJYEnxXUtSWtgi0EaO3cGgwchSID8BSWy8zjKzpGgQLSEyZQt3Ux+Io
CokK4ByaEfA6VLbxNAebmsY6CHTvOk2+RwqOeTcltbjVMz0XUbaIczuoNUF9
1rrNT4qTOUwW5EB2WUAMZH8kLFlGj6IY5g+7u6EY+5aNnMkwiy94pElY9IQX
kQ+1Kf3p/ZcokIeSgSObh7gMKRE91kUL6A0bHrIOsEKLGAH1U2RTrE1LWNiU
AAS4EUiWWFdUPSWWbkWlBwxIZSX63eBxOIs9235zup7HFBML1PnggCbj/AS5
lH0X9htXQSE6YB9lzBlFnXq1GZr02o+kH5ivB+1dtg3hwvn3iPso6kiD6QxG
sCXpoPIDowvIAfKg6AHoD+gi+LsDHbTrEcb5RlDUPtsUNR8gcnXdQUao66EN
kaMhEcNSnG8QRVm1Zh44o9pFnMkV5kOwsp364oS6HGKnL1BTz2kRcYHEp8pF
NN5BF35yJaP7EwgXfjJEYZkPxKpiKLK1XNLbTuWNaw7Au6CZxeoLUupUpIBt
gA6RfkF4+BC5aUoBig8sYPLgY9BMgJocYu7UOli7DmUhhHLwM9Qu0rdlU1XE
2ABJh5A3I3tYCJAVgZJK2L0WjAZtqqnagMlSsfV0vabGXGzioJX3RsQJq59Y
f/GqXAwDQtaBiKucEfaIX3VA6FnEYLi9cC3p5CPSjYKkl0rz4P/ZnIp60xlj
8YgsRLuIOwR/CHMIIXpLQTgQ04qMXkHnnyK2Fh2S2cKWFg9gJh3QTvDeICnA
oDXiihKNiBR2oZTaAc9RPsWBRCoQSy3YidyX8Lhv8fTTikV94W60H4SnL/oT
tW/X0fpT6y/hhmCkNnQU1tfbw6nA2lfe9aRXTiT9PHVZaWH0Pl7oHhpNJ4l7
w7rADD7pheYMPf5GakWBniUgAeh2T9A2iCFgUQHY7npIiuZwd6VCEsQAbUNO
mLOGQQS+JYTFBlO/gitwc5q5CuRXqevbJE+d7DZMvS8X1JCRSRACQ2gixbND
R3aNaH5eVPTHTuBKdu0+NqqUbpTmsQ8u5Dp9QJD1ECKFooih3zAM0JEgMvq5
YdTo/oHfR4oKE2ebrCE2FQK8MgVAbbMewXVmygbzu2DWDCAdqnyDqAPuG5hy
xJWPLSNeNLMM5LwJXSn2wY6H4Ijk5fztz9AXb7jIVAwdzeXFp4sIQNsbKJJN
r53+fQ+fPvZBupORepQgmyC3YrKnhBb4xz0ukXrPZ1G4QxmIjBrKqhb5FtgF
KIemDEyqDmSqlkrg0JHOUhmjLGHhVNtKkN0B88QZUBNbYL0rBXj0NJj49sI6
uf/xp8TJpLDALSd/gdxSOxSx/KF4E4Z4LL1Ujv0scq0wD8vmSk7B/0eTFS1o
UJkiAHJZhno2+G0WKo3XoY4AxekJGkXMjCpP0dcl6G6aoaKdthzEBAQNnS9m
2MKAggt/U0I1uA0du80usUaZZJoufDHt6WqZYJzxrD5PLgzQVLCk5PDs5NHp
TgSjrGpWKeKOR0g/cDK3JZFmNx+Z5BIznI3sg0mjj4BFt3YgUBrOxaIF19J/
qBUUYIpAZ2yDglpDHW7E6gC2DDS6wuQB5tsITEWd8DsjuAJ448xlSztPHpjE
tXk4f4CerK/lsorBARUPxQQ2sjCWX3QUeg4LpE+F/fS2IqTZnsqkV8qz9u8R
OxvS5Y0EmU0hGHkLjw7YLJZ2mlBqAKEAAAXVTQGqkQEre+CgtgUGLYq6B023
gevw5I6ypFnDvpWd6MyspmF1qN+AiSBpExHVW6iutk1Q1Dwr6qt54xuy8g6v
D7WPtQWB58d9WWN+n0fKsgDWf2yRVol/mqVptjF76cUzPuaaReA/tN5QpRNs
Q8VQ4M14vG1JFviodYSR9Uj/CwY7BbsEak0LJy4QeNXEQ2tr0yJUpsZDX//3
s2h4qdajH2nLCRo0Wz0tNLsceqJ3dzh+DW88dqCqj/wL87tFGrRUECQ/UBDg
9SiUV3A7Fqt7Pb2RsYeXkAoMi7CKOZQL4cWkUB19Gy9xin978uGFLwZ0lH+k
5CyKvi+J99l7kAVDOaQDUZf2OaAUDTYAPDrxBqE2nh4fol4aG8hbfIIUPvZg
q7gh+Grtjbwc4BhrYvKr5aKXnbrTOy59Vk0Jczd4r9qCoI/GfAVbhs6hdOPY
03SLhKUX8dQQP/Hh7Zx3pKHmEhLFAHUVHldGYESUgIIqzMRugstGTohshLhp
DYpjBIKVnvtS1cAeG2py5sr3aCLDZqspMkas1zQI5o0R95No1uNmmbjg0foe
+ct8RIeVal+Oj5FI7Dkfph/pfHk7uIqKkfOpk4R8bFZM5RM6rZYCzfVZX8Zj
aYxqs8HjjkFRJsftORpAeKiYA92wRGsacP4d6QauBn8DqWqYAmfGrxxF2b92
fuSAP004Brq/iXxulrH/0wBNZaYMWZhL682XM95m7nE+3Q5P9jN60vJHBl2I
c+nO9Ud7EzzMqmAcYMl6Ih15o7NKAReXMVVVtubLa4uvGRN61Xkakx16MoOA
83uRApxCHZJYzOCVgTkGITgaA1SLmlrB0GThFBr/cIv7Yi8+4ftToRhfL8km
XIovFHOmr5DF5AbtX84E5jWAIj+J1ZQ/4F4Sv7y6k3v9yJ58WiHQBIpLsWef
lQ1KHeBCoAkcGRryM0SfVICvQQtvKuuJLZ71EqXiXTTcA/IJUJio2IsvhXYM
+OUMmoiv2iPPZh54QimxQQ8nyrQ6LusFW4I2NCTUpU63+2q3kidV6s0ME89T
Apboa45kt8p5boPazC3fD19XhfxbYwH90Kbkf6qLYY0Cb/1IZXvVj+zpFcMc
Ly3RsHfS6RrrpYwzmKUoGW/QyFcTC9NSDbfE4AWQMKkbpmwUxjC0WqCHDC6b
AiReuuvlM0kchlk2PkhfYVGbxUvxraA9c3lNi9DXk+520Sy13hcsk8vFtZ83
J/hRuST9+HImelxfJJcar/K0vUMuP0ou5lzi/6jSh1x+hlyWODkY1kMuP0Eu
ahEOv4XDHNc38sb1oS8/l8+vD335MXZMXgzuO+Tym+b03ikQPHO9j5K/mNnJ
nyWX534LfsKwmPvX6Zy/fOEbnF9/L731/M6bX3wv3vzGT311fu7Ze+qxaarj
O3f9+PqdK8d37vvxHQzljV9ZjV9hNm5auxcH4sp3XNjp/7Xq77+Zf06yrh47
7fMbItfnN36+8zse5vzOuZx3QMxH5XJO/qfLftebv+0bJ90bIYr9v/Zwv/UV
39kT8tYwcf7uT60/67j0l8aT///YNXW7wPjFPzD/nIakL7uIi4Cw72xe+yHB
mhvz09N9o9KTUpSgELN78Scv8tPIXz3aNM+L18K4E0IVg9DefLFYcEC5e3XW
DX64sdXPWzKtJ96Oy9lrP7nAVPWxf/HXrnNuxpe+cV3C6aNj2A5v2tF7lhJf
PBevfwH32o0j41YsT0Z4ybMwkkv5ojmjku7QvWIGpYO+4F4Spy9VF9ISCTtm
X5vKAbZRPp5fS9HCfpexe9U/pKfTy7lhnTt3Hp8XKLGNcsq6tP/zxb4XvS/n
8sVv05xqlw7PCyUOONAn92JAJEamxuZVoCw9j2P6ov0EuOUzYvXVERG5Mytf
u2vpZ2zGl0PN5tUUzZT4pnZ8I6Ya7Ev2l5G+fLUdU3E8isB88tfkz0dm7Ji8
GnWML+eGdW4Uvfu1EYqYv3B+LeCoR0d+vxu+NK/Q/g7I8+Hlr1Ce87N67X6Q
2pxej2kGbB/iryaHgJPbV+SpECfnHcXJLvnFl77AJ9T1ZxPlMYTOM31YfXje
Uv/kaf2/QC730Id1+6JUF+0PXwUcHdcloHfZNrXwhv2upKU3Rd7RB51sdpgf
19s6o/WFwOKeymUm2AOND/rQma+AqsNSnmmfZTIvq9Sbxe/qsGT/p2i8G9Fy
S8ZffMtqF7z+cfj3H/MulxCa5MHh62lPJZ9Hty4IqvhYLtIcx/umZMw2N1Zd
fhLzYku/p7KdAPu6mhKacuQfqsq01Py4XkQUcPAnvbE7tbhAxcoZgCT4PHa9
jRM1cbFpMk528dseR568V1U7rlPCszzH9tIxP489CQqVhWIcTza4GGTfKepr
eanAZOuBIACQLM8BgDMjHkLnWz4GLG5wwzgC/x5PY8bw8vqwaK9c3o+cNO5u
VMAAcIoTbdYkFGM8MxU301LXYTEm7KRs5qdZ94MmXJKENqS0LiEHeNwTqwmN
cONoqjP3JZSW73S+HteD/kWe0L8GQTSEM/oa4wkV0i5LIlCv65GeBWrFqP+Q
jdQWSo/R5TurPXg86IBd01c54VXUP4rho8nhZ15FjWe59FR3DnQ+PQzxk1Ao
oNrjmZF/Ga1GFXOqAAS54BOo0slD+2NjICHf2Vt3eXMc8Bv+ZbJjyp16mJ0c
fcjZyPiJeZViZ0l2DOEAFENCH+wp+P0k2jGWC42HT0wMoxfmSdVgwWJI4mAP
7/LqBX+thuj3x7xWPbDywRRn7/f9xM1zXvDxlGO2YX62NYguZx1iNL34fWhS
fxrHjA+QbK6S07lLT2diTxySedGOrdZQuZnkQaYtPrNCIGM//Yago7e8Xr4G
ahSBOgcm8+aF1gYbZ3/wotAHnvKD8swIwRAkxqnaicxk1xApS89l5pDb/xQ5
q8s6mTng4h+CZMrVqFV+RyEOCf3PhPHr7ce3CALH9T+WY7Q8NOInBs+Pv+xA
jP8HeeiLmerJNCT6DziV/wDExg5AHA1sGgAAAABJRU5ErkJggg==
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 5</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see that we will need to set a higher threshold (which makes sense, as you’d expect more UMI’s per cell rather than unique genes!). Again, perhaps being a bit aggressive in our threshold, we might choose <code style="color: inherit">6.3</code>, for instance (which amounts to around 500 counts/cell).
<ul>
<li>In an ideal world, you’ll see a clear population of real cells separated from a clear population of debris. Many samples, like this one, are under-sequenced, and such separation would likely be seen after deeper sequencing!</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-filter-thresholds-mito"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Filter Thresholds: mito</div>
<p>What threshold should you set for <code style="color: inherit">pct_counts_mito</code>?</p>
<ol>
<li>Which plot(s) addresses this?</li>
<li>What number would you pick?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-5"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-5" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<ol>
<li>Any plot with <code style="color: inherit">pct_counts_mito</code> would do here, however the scatterplots are likely the easiest to interpret. We’ll use the same as last time.
<figure id="figure-6" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAZcAAAEACAMAAABI28vEAAAAS1BMVEX////X
19f19fXh4eH+/v6EhIT5+fl/f3/8/PwyMjLv7++amprp6emKioq8vLzFxcXM
zMyzs7Orq6uRkZGjo6N1dXVYWFgWFhY+Pj5jyzDVAAAACXBIWXMAAA7zAAAO
8wEcU5k6AAAgAElEQVR42u1dB2LjOAwUiyiKpHqx///SG5BUs+W4JZey0t3u
Ji4phNAGAyBJnrhkwpPj+uJLJfrZt+jj1P6P6+ljluo4tK9Xl+mv595yXF99
PXv3y8O5/EAXcyjL/yUUjfjqmVhMHGf2P1z8WQ0w+XH9D9eJ70dkOjwq45/V
ld9VKSmP+/3dK79pr3C+/u+NX5GHXL5XLl5ntIQ8ZPKcvshDLl8pF1IVHS0a
1/S3DJ/kx6F9p1y0t2KT/VJ6RgYOuXyvXGK4RqJRwao96Pe9LTtO9uvkIhPt
zqdzlmSnPG8nUd2Xi9T6gDa/UC46UaJKxLksihW+md/1+lqrAxb4Irno4GIg
muRcZsUSK/PzA+UZ5o6T/ao4Ofh9kzS5SE/5KOhTZfPxlNyvGhiSaTJFdMf1
ifoSIjCZsLFPnEmGMQZm9/RFz8no4WS+LK/EnT+m/mNxmpQh/xh99n6J3qmc
O7zMF+EwiRoHfIIDLiZ90Xf1Jf5/yOWr9AXawk4Ikfsxz0fmz/x+vr9Aaocd
+xq/r2MIJpdSgJzeoT/MKaWcXnCI5gv0RUZEbBKLWuRyR2H0RnGO67Pt2Oe8
47gOuRxyOa5DLodcDrkccjmuQy6HXI7rkMshl0Muh1yO65DLIZfjOuRyyOW4
DrkccvkyuchYbjsq/z9FLqH7QlLR/2Av/xy5gB0rL4kZx/Wj/MshlR8nF635
0rd5XD/Ev0jw/Bk/3MvP8i+hAUOog1b28+xYCMcOubzdx+ebk7aP5cc4km/v
R5Y7QdSR738zn3+Whd4o0CGX75bLIgy1mpyYH+HU9/p9+Gjlzvl5SNx4Guup
b+J8pB/fKxeCFx3jYqyGTg/p4V9+it+Pcy7yKmeJyGOmfr8f+bi+uv/F/8Vy
31sZ2pDv9yMf1/+QV0JBxLlNZrmoF+wYmpk/1Mhkb2jTIZc7+mLGDC8iO+Yf
eHqeklLCJDujGnXoKpNxOlB5SOKp/n1O/ch86JIiS/Rrfp9/NEDTd/yRVh6S
eLIfGaMWK3b2cTId8fNy0ftwGFevDzw/5o+ZKbfE6fFX9EXfgCm5cKFpOcwH
OvzLE/riR/jy6X6WyYPzLR8PB0Il5lCXJ+SiYlVxwcbkG3nlnREL5pDNE3MV
tgZIvpjvy/teR9ljLNbX1sXknarYHB9vBKTMMfL/K+Xy0EAY/6LVK0krudp9
2b9MBPg0uWyLNLvaIklVYiV5m8EcK4C+Ul/4Y6eo5Bp3kVrLXYHofxqc+Sy5
IH4zTNyWjM9QpjmxctEDGcaay935mv/wpKz/S18gMzmHBk5fgzE7wfO/nNp8
llziaN6bB2mW2diK1+ZuGKc5GID/sCXLP7OS9kEAJS9VYYX37F/in6ZD5bu7
KdX39SXJg1q+e8r86Bf7iXLRIL+c89Edcvlp/mVMa5aO3yUXffTI3NCX/M7h
/w9ykYdcrv3LGbX2Ov8+O6YOqeyecnXGLOv6++Ixxw+57J2y0EZx8W1yoR0l
h1yuT1nm6hn/EtPCGcmSkdgcFpbKO65EuVfLBf+aXFh7atvWPuNfZCipTOcp
9YxGqjv+HfTnXYDzWCR/dcoNqPvjmDUP64vSq2HxgXMWBAJBMX0Hl9F8X2CH
xlwrRnnnXO50/pkFIuZMvULG4EdV+fqUOzWM9N/j/sXv4DFTA5OpnDdhHsO8
c8ByJyJWkjPGD0N2ZceS1lrb9o++IzDAhI+rJXdVYuTEdrnn9G9GZIe+7Fml
ewY+33uDmpgtcrJP8q6b2K0ghzcfIdlOXjnS/vdn4rEp8eEKRmyKAbTUh/v+
TLmcSiZ24tdb75ATAZD2icVKsprURd21YtET6dvs8afGyei/i4+NYrMfWd2X
pJzL+/qK03LNz5CXRsoPjmn0TRhZ6YcmafxtfVGJHSwyS+/G0flyyrVfj1yq
1Tv0rZtaLvQLb860fuC+DgN9kn0O+eN7yuRfr7+ckVhOcXJT4gVDcSd/kYSn
BZYrN5qW8Mo5lblof9FyqxhhjF8SKH+72vJYGKAD7PB37ZgZzTrKcnhBUawX
8p0v7kzpM8iIvYCPpAT3R0k95lozZq7OdZ3lR7kk/NaZ6sdpAVr+aX1helrz
jouh33U4nb3TgR/vr/uR9cIK0BNjL6Jf04diTUzS2jGjt0v/Pjh/gDrmkV9E
8T+F3+zEY7RKNDgReFzSF6NMN+q5bnYRCsjlZp/Iq7FDY7ZMFzDY5qB1cE2u
VhSU7aE1gj2QaGoh+N+Wi2ACl/cOBD2GF9A/8hazybDQciwX7rieDSEQf7E5
ML0JoOHxZVQuecOO6Yfyf87lX2oAyHf8rI79enSY7OwSlqhu5FODan5dRUH/
io4Zi5arGEpHQd84b1I0XfJN+76MoR2v1FXU/S9RyfO9CEhPfccY2HM6p3E9
strtr/TlEn15ckFfmPvIUUtJvf3XPXzaV/jFhdPh/xZNZrfK5XOPfAmT9MzI
38n39RLPemVbzhLHXquPAijlZ2GotR5IPTePbxplqJbzL3WU7/EtgyQu8kd9
A08OQ/2CT1G12AyUI7LArVgrppFaqiS56IfxZQKS0Eo91D8ul/n+VivOZeyz
k3v1lykp9EIxU+f/yvd/EAFPWNqluVpCB/k4P/cfqFe2FtadXeURN+svITnk
fK1Yi1SZu1uLWaxd6RbUcwdg0/+uHRt5dZ5xmB18Kr/TUk7j5HC0ZaQ6ceNc
xT/K0/X6+E2Myci+bc2o/qf4ftennGubbR/WH71DX+DwlGYKoVfpEE/cR3Xk
C6WQSk4FZn30Ja1xmA5jrZ7mwRIMNiNdRs1LK+Re/iFn66Uv2i23ipQkh1wW
C1/YhBXPysXxKS1cdGUSgzIL3hbjaLPBaXYloI965eaydMLFC/3Isd1sRU5S
zAcPjE23/9Qyaar57M1+f5qc+IKHXKKfzudmi9f4yfyqQ9lLx89mEj5r1T5a
01Oj8o0+2mOuwvrqwbc8j+fzm7xxPY3JSrDAwgQX4uWyUC2937/JzdC0kOTo
E1/Cp35sqrLSr05cDoqhHMQR8DRmVm58wijnrrCbAZcSSaIPuSSbJm7zbt+r
DhOS1GLWFrALwE2Qn4tPGLHz/SqWLJWDQy44x/JE1wsT/WegbFICVbIrnD76
cho1WgdwUlXM7Hkp/U93We7UK9mT7yA3XtKf4BFWIOQG59fBLumZe+GHKc4Y
jIzAi1wVYlxFfTTyX/T/13WxkT/Jg124Zv4vfk0b0osehTgrksrhhjb4p16m
9fiYAGGbn1miDn1Bvk/8seZpBFpHyYhmOUVRt0vft5pf6U9fMZ7s5ft67k4K
H7l/cwDGtVyGccUfe5xp7m9veTl8h+v1YNHo8ANffPH2PD6aTE9HgyfnSb6H
Hbt/BPkO/z5GvyA2I5GvxPprOBRvPKCs+IzNyMmqhTaAmk0VypV/SaZhmOqf
RMqu5OKe1xe+sMjk7OlF+Mg0tfPpvWI1v2oTo5SmDIsVgpnTNTPbu0PLIx7z
V9M23eNyoZnkldFLUklhGHkn5wJvjM9FNb7iVU5lHUCafJ7OB/HpWOPfFMbU
YceiQTGP4zAbsCte9ez317EZBVYslMikYREBaOeSJkLiVcgQqpY+tpbysGPx
qNnruKWaqjFXV2mQafJIwnPraXwRHvCq42vS8ugXu3IWlO3n7eubfBSToUF8
ZsPKy4wGTD9+bHx/Ul/4nTwuvzvgpY50pRBHeFsmNmg+dxeY2ZaLc3RX7p5y
M3w0VuE+bulT9KU4aVaS2DiiMOF6Lte41RNaHnK57kuylrYkvYMnR74ZuDBe
EIamiK4RTc7kBLrMZP3QMyuXySaHXDYXiOKJeW/+mOhVLOw7RcTmSqmahfqy
8LOTNZtZr0LpFYzDk2Ne341+sdyFYvI7+mJmO8Zrn7lIPzYGxqoRF7wzNRkw
rkuzqgQccrm4egzuObVv+H0t1o1g5gbPVkZLBv5sUJiGm4ipqUNf9k5ZNO18
U48nvICWJLu5GpxfYmjyallcMpFfkJlQxqIXPlJo1qdh/tUqEKhXAhPukfE+
8rIHSd7cMLe+KeTvlUtLhmfSl6qifmTrQMGc58btEvC0JN6YNougXOU2+sKr
eiXFch0k86VTT8dGtSv9ul0ifWwn0C+r4uzxYNcPm5OvYFL7azzl817rPrCt
viaobEbJeFW5hVrmW8/qpXPYbA5TJCt2k95M7xc3qM17q3ySj/b68etVTb9M
Lkk0VnNj5UlO65FVe9mPrGarFu55+ttXIbUn5vNljN96UyWPeabyMKaO3Utq
MmRsy6/dDfkcf6o6oZn67XXktGLZjCfT/l1IQp9mMOCyX4xvCN4kFxPPjEvW
yH3+hAkYjCtnpdAryazkwm+UKx2/WHQFVdG3B9L8frkkLjufhxmMEdGO5bPJ
yrflEQeu2RwmeFySmr6pzM8qESIANSnNapj4ZuiL80mMS94YosiZ0HdooL/b
jk3VET6ELe9oG8vg9xcA4HxxcsSN4FSknBBhQ/+4ihJJvgqD9OZbgMLkFgdP
vXycGfHwFt/kap2SMR8uWPr9fn86Dt8njn7k3IpznrvtXlF5yRlfTbBgm1Zi
TZbM+FrxShvkmsXkq181LCBZNVFtJOH0YzzlhwLgXx0nL8/oBysDSzmLY0Mo
M6W52MADTHkzVKeaX9Bsu8LkRAMI7THKv3SqW8pnwGb//ebvuTtwQbEf7HPy
W6DjTYnlV23iaqkQOK0rgC3B4iMubqbIa3MyC8x/FQWLim8GMyyN6nLesCjE
A/d9mNLI9S8dpHlTX9R4Q+vz/dVhMTuEMOAonDdMpeNOMBJPezW7jQnwMJTY
MCuEWOZj+wKAlu0kUbXeUCNM+dhvh/kBH49A07+oL8mDZPHfR97B4vwXIoJz
FMWMUJQaQnVw9wte13o7UdGDyVTQ50ZoXsVKsjd3k30jbhmKnqVppW8J9LLx
YzXkarTZnUYPkghXN0s5Uv7kCtz+/DF9vv2r57t2zwWw0dR+BE/MNOHLmzDg
2kSSUsKUIrn4uK1GO4xkYUgGzQOUzjBNhYHEDwKqJkb5MmB24ck82oTzQYXt
RxPTr+YnZ6chG4f8GT4MiaZya34FRc3QlQSoZU9CEGZKFZkHNKOWBOyEtXUZ
9YmqZIztEde9s1CPWx61nvf7C+csXQqg7vLO2sY9M9faXIWsdeUzcl7TBlgK
snS1SmIwrGdNbRJ9beYglq+OXyUbSjPtk30mJJO78ObWvfwi/+LWm1v1I5av
8Yh+xZZ7lYBIcs11g/Z9T0R2yzmZsqrUPJ5pCgqmEFqsURgV/InWqwEM+qG+
Cz33QMn9Dcw/fBjmTh2ZGsbOT8xPDsarXvW8ckuiSPEgTJadoMcFheGRRsY2
96woJfXI6np7fDzMz5Rr6r9M/voejH08+al+ZE3D3TgSwKk9rK58OxiHZiCG
Zj71nLomes8gr4Vv36tpnn/M6Mty8TrT1RrT+/YXH8Jtdv/sGC2u/zbvoqEt
MI++Q8UamBIgjSGWak2InA3Afm1dGzEyI1fbdfF6IKFlJTWMHXXI+oCjik8u
SbjAagu5nZCtbzlxmuYotJJ/eN4F0LC8fCYeYx6DxL2veqMbp6nZgtaEUT/M
wnWlHqUli/FVM+wx4wvXT88hWZREtR6ZrTDIR+mPZvpy+bf5lmi0GJJn9osp
zw/n4Q+r6prci6ZZWUlPYTHXsT48TbpcwWAIqgP/QkYuYT3DDCrqlppGLM2d
uSJh++iR1n9532vTZOfnJpUSMtb62LoUdcnQBugaHg4SqQvNFPNiCA6Ir8sv
M30cJo1ta2LIRREf6NrRSD/IP47AMgRo1r8eyH+J1+eem3fhDQuZJDRNtkz3
guJiZgrgi1g9Qjkl84yXuuTJVW64VOotX+Omra8PzEEcmJlec3hlbueR8i/r
C3wtG5+spPl5IxVDPKuCa7JCtBO7Jd7cUKiaCTxtAilWVzOwFo66Z7HpQnBA
Mm6FlXhD6JLkQ2hM/+k4med+rkJ+1s8h0PDzgkrKdeLBXkU8NEpUrJhueGBf
PXBMPC7WOgO/UvrMNIld0BisDW1rvR28Kg7Ly+0LRl8OltN/U18Y5o3jZOoH
/X7I/RozlYPrxkwDwlVZtU0YRmpCMYDNuD6W+MgJX9AREQscJ7JTLHp2XuuL
YkI5SypizPV20LL+I50a+dMT2fK9O5gv3S51szRc+Pb9gnqUS09aLn06Q00X
zqntNxOTY2Gow5iSVUoucD1JSq3GX20LW1fze//APPhbcrldVt95h2Eh+EIZ
qrbUYUxkVobucOU5/QT+ozew8RlJLVhVsX471ZSmxJhQGlCetrTa08RmnQmZ
jZq7YpF2Kio2mz/YyZQ/va7rCk9204n5AnLDyGaRXIjDBFsmyla1BAlUrAbx
mYSl2Ar9IhYMr+NdXoUBwWoz5MqJsFKG6d2yltqg33+kpSl//x1wBmvrwuDp
ERqnpV9UIhAnN50xru1rj7CwkKtMhLLeD16UzPv74IPqZM1XAszmRNVG+MbJ
NQk9CcN8NxNk5N/on8mf3m6X785tlQsDxrVCU++rp8nAr/SuLxmrQgAQ1MCV
eiYya25Qz0wwfyHwzGEFV4SaZLX8ZS4lKDn39OsddvlfADDz++14j7yDWsCB
UTYwNTD6zN/efnkccBleMVc72/iZsVb0aIxFdCvnXQyuEhAEspYqocINMhxj
96OquLDM+BhM7oL7fnyzOezYBHZilxhNGC/BUmoRLAMntjzO7FOubZwC5s/g
CMBbAjPDlL68TPQx5JRGOF8nM76/wyBqjnoR0hp+PXhLr5mai/hC6qKMmcNE
turgoBrZxVqHPy+XpLJuWvqa1D1xxxUyQwIqS9GWAXVmNWWcJSxa6ZwUgZfp
WZcsFsqwealtjKdmsvkAQ3YjI9aC2o1advuB1r8eeL51+nFAg9K/MHZ+WC7y
IxwGLUc4LQDtGFzR1gnrUfrCrdwQTawFLIacBFtkAe0bBGVla6zRrmw6H4tR
s79nN8c+s7ATyzWXVFggY40Lk2bkwlcyasOYmfcCeX7OPLhBLaVkrX9F2vkJ
ciFXQrkj3foVrA7reGIrqjZS0NswCtgQLSMPYbw0jU6mZIdRfQzOXCyrd0hM
lO4gEZ0PstoOKpvmy4Zd8HqlJ+t6JnNbfq2OluxP6ctcjuI7dTHpwXca30rr
SaS3aaWCBtTeRInoWBjrAfgL2nUhS2+dGt+nXDmAnas2DHDOxP4YjOBoTLVa
ZxJZGVputs0tGwBh6eSiS9Hg/Yo98g9N3Z2mVMkb+198XKWQfPcFJZeGhVpM
Qixy3z1RGVVVYCMxgpzRUUEZZGAiNW2YpVRRjcDbNapjliJZdypFuIXFPjQ/
w6nmYbvs3q5YRX5thV9iY6acgzdOWar8E/qisNGSNovKuN/qoqkROQckYIaW
ELEyqaid2YQ/gLo67ylahGHOWdfZGsxY6/PFVjTGSEWoC4gVCvWV0hrAazjo
uUU2AKITEBpSHBXNmVZTAdQri1mmBHGz3dusYn+aL8MZo/+K30f7C5/GH+VX
bRbw1VUGf0BpS91BOiWS/KKBj0AVxTLb0dpPxlwDX2MqB6aM5zHhnkceTxk7
QoXEFm1bAq5kpQ9v6fm+n3hPbThxv8dJXu7r0csWze1akx3KzCVa+svt2GlB
Rvj5Wl+QFgqfRPaF4VqapO1QOUbZsrOJTRGNVRpupKsrPNvxzrt0vze5ihlI
D4PXsRrsf+cnMdbR5MhpZnnprZKPpquLMG0+dqN2t7/L9cqnas0jl79aLnTy
p/PZp+D8ej8y/bo9zj/RLVB8S2dedAiXjWJdr0RrWe+oosPqqmp5TXg9sz6N
gVaUSEhLNMHoFo0ABq9Somkc1KqKzNgSkiKWswl9m3SYIpYFyunen3ka17FC
pFqGyEBdb56Xv1ouNMKFjZ6apxTfK/1Xha2Bp/Qg6FdIKlGWTBzW+9RFA3vW
1LZzbUUIZtn6chmZqqasGmHxH6pieMIiaC4bhAwoD/QlUV6QxxgBWVKZDDfE
hIQhVSKWHyxeuywx1cuQjI205MKcNeE3+S0Vgfyh1AW/2GjV5F+ufzXH2qIk
W9Sh+NWloF4WTPUWKX6lWwuCTNYygCqZ7XHGJJHaMnh9JoqmTvoGJNqGmi8a
P2sDNwBiNFRxXFtrUWNgCRqbG2DQyo+WRURHxA4qUy/2CZCYx9qknLs25Wq6
fOL5oH4iioxDTPjPVpj8kXwSNt+NbVzqcr7Rhd13wB1rRF+uL3pe9zVnbQlV
QNk/LUqydBBFWdZoRWqrouG2hxdvK1uVFCn7GBiPUE0/qZ13W+AI4iv2jato
5YXzNQKNHbGM5JNMVRfGN0RybPCt1NUaejbRNaa9joa+hvrF+kI/e30+5dlH
7zBdq5SxXQl6XsooKEOuYhmviw46UdWdVRKfdylAZ8GK1tUN7FvqNDMchTIw
/1gHF5SUvQ8HGiF8RFD70AyggHE9tMYQp5xxPzl2HvSnWXLZQma2xWSfW81z
sqeZPyb5ZX18u45/lUvne78PsykOru8A0qsmq0lHXGIH0GFs4fqs0iwDXula
m2Fak+27OmNlUzcQVdX2tkVA3bGm51CnlpxHh9SPtW1boUAAK6YbMk9gCDrX
wJwylqjtXGGquVTC/5SxcVYuwxj1enSmx8zkPOJE/WE8OWR3bZfCqZcp5FPb
Al4kK2rWpJXosx5QTMkG2CPOmsZWbYEnGwud4sbCzVhLiSb+6anOTON/bAWY
jdGMP9G2PRIh0J0bT1KjnJD3VPmyoU8gdPwn7aorWpK795eU27aMMGdrkov+
1fHYA4xmXsATS1tgXqkqII+ktLaqbFrUpi9q0RStshaAS+qaGmlmY9POFrbt
eiiQLeuyKcD3Q9WEg9xMpHPUB9oS5AwRGtd0VWE+fB2KylTP1yRHX0OeoC6E
EHFnSSXLjxfKXSSW+mYSuu7ClL9OLo6q8s6mqcB9nCLc5YUFcbJJ01ZYOP+6
wQdl1lW2KEthIZeCJNF2BURleNb3TdcVbdOXtq7KtkCgi2itqduMwrYErsZR
WZkQgMnTg+dUBJKM6gGwKQTJjmPGKRAWMAXw7ITdaFFdZTb6cm22vFE7l6sF
w/L36QsQFpoxwoohI4yr6MC36OHZVTnA+0M7qtKmPR63YE+0bQFx0MCmLsvw
UQleWZq1tquNY42AhJBnImooDRIXGDKBck3HNMRJ5o1W+tLsf9ChSFw0MYgO
DZBcM7HX2NUR6uRpxs+W2an0r5QLHVORoiJpuqwoCSDDGMaSpSnSyqFjeBRi
6IeiblNEY1nXwNpBr+Ds0V7b9LbMbF0UHXQGSJrtRA/1QsrTC8iFLpIEK6oe
WlZZGD6IkkNzVIN4IVA2EZ/P3c7Veoem2oyWW804U7cls+qDkr++jowxiwXi
JIOjB0ZZQxlY0xYZ6pE2axx0pGVFOlgLEUB4fZd1fUr/QDrwQhBNCnNmG9bD
E3VAnIFhVhKZaMNS2yB2hihaJJaY/gPyADQlASJQs9aGApvQYSmGEhO3nOho
ZmoccCxgOobvjCDRl01Oetmbon0BQelvIaS/K5fpBy7TDJJBSJx14BZ3A/LL
BipgjC2Qv0BPeptltsqgEmnRtVARSx4GzqUYKkioKCACSMSmA6TXF4gb4Pkl
vA/cUEMP9cDZuh7eDOkpqmf0MKNhjaIELk0ANQcuysJJt34HAA0lc9HFAOFx
m4JMFBtsYsIvSLSBPRprddiN/h1M9E/RF0uwFyuyAeleWUBC3GUZsGVjB8ws
baAqDMdPegENSQsEbhAKlCTNujTFJ3gH1KSDHqVt2WdQGlg3wGdl1kJ8fdMX
gsNNdQ2wT4QUwBPwNaBGDaK6BoJpKairHSPgh3o9OE03cz4pNdRai+qlmPNO
40WnUBxSvruAX2HQUyeh76Yy5lvwmk+RC3xIQZVG+P4CuEuaFUhgKhx7A0OG
SK0csrQvIIKWpILYrcPDBUkG6jXgoxTmrYVA8W/a4ikLFnODEKIr4IrSDqau
S4F+Ih2qIVDYMPgwjjpOlwpmEV0LBAtp5TpQ10wFDgGRb8TcPVuKBhJqYm96
A3gbaBynHxgd0BfDrsQ02g9WTE1tt7+Xp9QMcCf4bRocNewZ0svBGlYUAwIn
CKCDfyHHgr9wrhAVRNMVXizk8SEx/z+egIz8Ra/BazsSH96GhAj61LeQVQl3
hc+yFm4ICtdCg9AqqFugn1VTFgb0KCJwoLMTwV5TIR9F8rp0qbnGGd5Us/cX
1doox3kpcYidB7N3pt/6t1azqs3jOT9xE9pnyIUyuqqGsmQ9TRXJhqylUABW
i5e2GFIGrwOB4LDh71OSRuZlEwSUFtMH2fxv/GOhZAU5pt70kAtJA76pA1aA
TzJYvrIWeKJpiZPO6gb/wxLasmz8wHn0eCLggEoAHMUP6Ic0Ef7Z+nGnBocO
+m0nPDeKGqbn7gSFOSosTCCaukhmec3xnJhWPkjhtkmRVj9CLrqCRSqDh4GL
4bwlA4XAiQwVymIDogDYoWzoesgunnrmjZj/QxpSZLPAgtC89ylafMGUlKyD
4pAuZfBGBTC1pkGcUCE5SjskknAwDXKhDEBb1deoR/dlaZFCNc6VTDIwCxGL
VB00CUtpkV9ZfzMxmDEhPGmW6kVxazA1tikqFrBmmugoSqX9zsCAN/DLuFrr
q5jb/Yw4uSWFINHg8CASh7gM59rrHgdJTn+AgBpSqC7IxZupNEuzSUWKbHuR
APBsEawbKR3JxcsLQA/i8R7ysWQDM4AFiMl7hAGIqXlfNRUKOtCtugXsICA3
eJwU8QOMH8LyhlqiqOjdVH7MaQOWjW9oA7Ra07BB3P7IYz13Y2LZAGeAXHlk
revd9HM1Z4VaT6qfoC/+B8LJQgTC5zI4VVD7UvicVFj4dWT+ZMUoEIvakF3J
YecRqNikWRBKF/wPeaNugN6RrAAlpLBu+AapbeGybAPgDW6s6Bsf5ZW8saYZ
EJaVtmINACKA0sAOYKAagE5pOOgAAA69SURBVBEI+VBdaJAYORdmaxP9pkYR
jYsw4LYlDmkLqcF2BTeiS7dRiFU1YT00/QfIpSrSlkbB6S4dKCRGMkNCAohZ
paQibYNwGQ8Uu+e/e6WTYCZHRH8PmQ+s8T3CY4jV4K8G758sHFeHylxPjgyl
NA7KASmaAHBdtALCAwSH/cKwd+hfK1kBVIjBt6D7vG3RvcZg9ggtYJZMG2YM
lzViMcQONDGlCiQP4Vs7MR7S/+N8MzUqq5E+SjlcIz+tj/Az7Bjr6eQspohW
LQ5jGIAul0MxkMuHviAUzmobPMYkmeXvOQ7YSsN/GCSSDVk6/1eQLQxPIpYb
Cm/YEAJYEnxXUtSWtgi0EaO3cGgwchSID8BSWy8zjKzpGgQLSEyZQt3Ux+Io
CokK4ByaEfA6VLbxNAebmsY6CHTvOk2+RwqOeTcltbjVMz0XUbaIczuoNUF9
1rrNT4qTOUwW5EB2WUAMZH8kLFlGj6IY5g+7u6EY+5aNnMkwiy94pElY9IQX
kQ+1Kf3p/ZcokIeSgSObh7gMKRE91kUL6A0bHrIOsEKLGAH1U2RTrE1LWNiU
AAS4EUiWWFdUPSWWbkWlBwxIZSX63eBxOIs9235zup7HFBML1PnggCbj/AS5
lH0X9htXQSE6YB9lzBlFnXq1GZr02o+kH5ivB+1dtg3hwvn3iPso6kiD6QxG
sCXpoPIDowvIAfKg6AHoD+gi+LsDHbTrEcb5RlDUPtsUNR8gcnXdQUao66EN
kaMhEcNSnG8QRVm1Zh44o9pFnMkV5kOwsp364oS6HGKnL1BTz2kRcYHEp8pF
NN5BF35yJaP7EwgXfjJEYZkPxKpiKLK1XNLbTuWNaw7Au6CZxeoLUupUpIBt
gA6RfkF4+BC5aUoBig8sYPLgY9BMgJocYu7UOli7DmUhhHLwM9Qu0rdlU1XE
2ABJh5A3I3tYCJAVgZJK2L0WjAZtqqnagMlSsfV0vabGXGzioJX3RsQJq59Y
f/GqXAwDQtaBiKucEfaIX3VA6FnEYLi9cC3p5CPSjYKkl0rz4P/ZnIp60xlj
8YgsRLuIOwR/CHMIIXpLQTgQ04qMXkHnnyK2Fh2S2cKWFg9gJh3QTvDeICnA
oDXiihKNiBR2oZTaAc9RPsWBRCoQSy3YidyX8Lhv8fTTikV94W60H4SnL/oT
tW/X0fpT6y/hhmCkNnQU1tfbw6nA2lfe9aRXTiT9PHVZaWH0Pl7oHhpNJ4l7
w7rADD7pheYMPf5GakWBniUgAeh2T9A2iCFgUQHY7npIiuZwd6VCEsQAbUNO
mLOGQQS+JYTFBlO/gitwc5q5CuRXqevbJE+d7DZMvS8X1JCRSRACQ2gixbND
R3aNaH5eVPTHTuBKdu0+NqqUbpTmsQ8u5Dp9QJD1ECKFooih3zAM0JEgMvq5
YdTo/oHfR4oKE2ebrCE2FQK8MgVAbbMewXVmygbzu2DWDCAdqnyDqAPuG5hy
xJWPLSNeNLMM5LwJXSn2wY6H4Ijk5fztz9AXb7jIVAwdzeXFp4sIQNsbKJJN
r53+fQ+fPvZBupORepQgmyC3YrKnhBb4xz0ukXrPZ1G4QxmIjBrKqhb5FtgF
KIemDEyqDmSqlkrg0JHOUhmjLGHhVNtKkN0B88QZUBNbYL0rBXj0NJj49sI6
uf/xp8TJpLDALSd/gdxSOxSx/KF4E4Z4LL1Ujv0scq0wD8vmSk7B/0eTFS1o
UJkiAHJZhno2+G0WKo3XoY4AxekJGkXMjCpP0dcl6G6aoaKdthzEBAQNnS9m
2MKAggt/U0I1uA0du80usUaZZJoufDHt6WqZYJzxrD5PLgzQVLCk5PDs5NHp
TgSjrGpWKeKOR0g/cDK3JZFmNx+Z5BIznI3sg0mjj4BFt3YgUBrOxaIF19J/
qBUUYIpAZ2yDglpDHW7E6gC2DDS6wuQB5tsITEWd8DsjuAJ448xlSztPHpjE
tXk4f4CerK/lsorBARUPxQQ2sjCWX3QUeg4LpE+F/fS2IqTZnsqkV8qz9u8R
OxvS5Y0EmU0hGHkLjw7YLJZ2mlBqAKEAAAXVTQGqkQEre+CgtgUGLYq6B023
gevw5I6ypFnDvpWd6MyspmF1qN+AiSBpExHVW6iutk1Q1Dwr6qt54xuy8g6v
D7WPtQWB58d9WWN+n0fKsgDWf2yRVol/mqVptjF76cUzPuaaReA/tN5QpRNs
Q8VQ4M14vG1JFviodYSR9Uj/CwY7BbsEak0LJy4QeNXEQ2tr0yJUpsZDX//3
s2h4qdajH2nLCRo0Wz0tNLsceqJ3dzh+DW88dqCqj/wL87tFGrRUECQ/UBDg
9SiUV3A7Fqt7Pb2RsYeXkAoMi7CKOZQL4cWkUB19Gy9xin978uGFLwZ0lH+k
5CyKvi+J99l7kAVDOaQDUZf2OaAUDTYAPDrxBqE2nh4fol4aG8hbfIIUPvZg
q7gh+Grtjbwc4BhrYvKr5aKXnbrTOy59Vk0Jczd4r9qCoI/GfAVbhs6hdOPY
03SLhKUX8dQQP/Hh7Zx3pKHmEhLFAHUVHldGYESUgIIqzMRugstGTohshLhp
DYpjBIKVnvtS1cAeG2py5sr3aCLDZqspMkas1zQI5o0R95No1uNmmbjg0foe
+ct8RIeVal+Oj5FI7Dkfph/pfHk7uIqKkfOpk4R8bFZM5RM6rZYCzfVZX8Zj
aYxqs8HjjkFRJsftORpAeKiYA92wRGsacP4d6QauBn8DqWqYAmfGrxxF2b92
fuSAP004Brq/iXxulrH/0wBNZaYMWZhL682XM95m7nE+3Q5P9jN60vJHBl2I
c+nO9Ud7EzzMqmAcYMl6Ih15o7NKAReXMVVVtubLa4uvGRN61Xkakx16MoOA
83uRApxCHZJYzOCVgTkGITgaA1SLmlrB0GThFBr/cIv7Yi8+4ftToRhfL8km
XIovFHOmr5DF5AbtX84E5jWAIj+J1ZQ/4F4Sv7y6k3v9yJ58WiHQBIpLsWef
lQ1KHeBCoAkcGRryM0SfVICvQQtvKuuJLZ71EqXiXTTcA/IJUJio2IsvhXYM
+OUMmoiv2iPPZh54QimxQQ8nyrQ6LusFW4I2NCTUpU63+2q3kidV6s0ME89T
Apboa45kt8p5boPazC3fD19XhfxbYwH90Kbkf6qLYY0Cb/1IZXvVj+zpFcMc
Ly3RsHfS6RrrpYwzmKUoGW/QyFcTC9NSDbfE4AWQMKkbpmwUxjC0WqCHDC6b
AiReuuvlM0kchlk2PkhfYVGbxUvxraA9c3lNi9DXk+520Sy13hcsk8vFtZ83
J/hRuST9+HImelxfJJcar/K0vUMuP0ou5lzi/6jSh1x+hlyWODkY1kMuP0Eu
ahEOv4XDHNc38sb1oS8/l8+vD335MXZMXgzuO+Tym+b03ikQPHO9j5K/mNnJ
nyWX534LfsKwmPvX6Zy/fOEbnF9/L731/M6bX3wv3vzGT311fu7Ze+qxaarj
O3f9+PqdK8d37vvxHQzljV9ZjV9hNm5auxcH4sp3XNjp/7Xq77+Zf06yrh47
7fMbItfnN36+8zse5vzOuZx3QMxH5XJO/qfLftebv+0bJ90bIYr9v/Zwv/UV
39kT8tYwcf7uT60/67j0l8aT///YNXW7wPjFPzD/nIakL7uIi4Cw72xe+yHB
mhvz09N9o9KTUpSgELN78Scv8tPIXz3aNM+L18K4E0IVg9DefLFYcEC5e3XW
DX64sdXPWzKtJ96Oy9lrP7nAVPWxf/HXrnNuxpe+cV3C6aNj2A5v2tF7lhJf
PBevfwH32o0j41YsT0Z4ybMwkkv5ojmjku7QvWIGpYO+4F4Spy9VF9ISCTtm
X5vKAbZRPp5fS9HCfpexe9U/pKfTy7lhnTt3Hp8XKLGNcsq6tP/zxb4XvS/n
8sVv05xqlw7PCyUOONAn92JAJEamxuZVoCw9j2P6ov0EuOUzYvXVERG5Mytf
u2vpZ2zGl0PN5tUUzZT4pnZ8I6Ya7Ev2l5G+fLUdU3E8isB88tfkz0dm7Ji8
GnWML+eGdW4Uvfu1EYqYv3B+LeCoR0d+vxu+NK/Q/g7I8+Hlr1Ce87N67X6Q
2pxej2kGbB/iryaHgJPbV+SpECfnHcXJLvnFl77AJ9T1ZxPlMYTOM31YfXje
Uv/kaf2/QC730Id1+6JUF+0PXwUcHdcloHfZNrXwhv2upKU3Rd7RB51sdpgf
19s6o/WFwOKeymUm2AOND/rQma+AqsNSnmmfZTIvq9Sbxe/qsGT/p2i8G9Fy
S8ZffMtqF7z+cfj3H/MulxCa5MHh62lPJZ9Hty4IqvhYLtIcx/umZMw2N1Zd
fhLzYku/p7KdAPu6mhKacuQfqsq01Py4XkQUcPAnvbE7tbhAxcoZgCT4PHa9
jRM1cbFpMk528dseR568V1U7rlPCszzH9tIxP489CQqVhWIcTza4GGTfKepr
eanAZOuBIACQLM8BgDMjHkLnWz4GLG5wwzgC/x5PY8bw8vqwaK9c3o+cNO5u
VMAAcIoTbdYkFGM8MxU301LXYTEm7KRs5qdZ94MmXJKENqS0LiEHeNwTqwmN
cONoqjP3JZSW73S+HteD/kWe0L8GQTSEM/oa4wkV0i5LIlCv65GeBWrFqP+Q
jdQWSo/R5TurPXg86IBd01c54VXUP4rho8nhZ15FjWe59FR3DnQ+PQzxk1Ao
oNrjmZF/Ga1GFXOqAAS54BOo0slD+2NjICHf2Vt3eXMc8Bv+ZbJjyp16mJ0c
fcjZyPiJeZViZ0l2DOEAFENCH+wp+P0k2jGWC42HT0wMoxfmSdVgwWJI4mAP
7/LqBX+thuj3x7xWPbDywRRn7/f9xM1zXvDxlGO2YX62NYguZx1iNL34fWhS
fxrHjA+QbK6S07lLT2diTxySedGOrdZQuZnkQaYtPrNCIGM//Yago7e8Xr4G
ahSBOgcm8+aF1gYbZ3/wotAHnvKD8swIwRAkxqnaicxk1xApS89l5pDb/xQ5
q8s6mTng4h+CZMrVqFV+RyEOCf3PhPHr7ce3CALH9T+WY7Q8NOInBs+Pv+xA
jP8HeeiLmerJNCT6DziV/wDExg5AHA1sGgAAAABJRU5ErkJggg==
"" alt="Scatter-countsxmito. " width="407" height="256" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/scatter-mito-umis.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 6</strong>:</span> Scatterplot - mito x UMIs (Raw)</figcaption></figure>
</li>
<li>We can see a clear trend wherein cells that have around 5% mito counts or higher also have far fewer total counts. These cells are low quality, will muddy our data, and are likely stressed or ruptured prior to encapsulation in a droplet. While 5% is quite a common cut-off, this is quite messy data, so just for kicks we’ll go more aggressive with a <code style="color: inherit">4.5%</code>.
<ul>
<li>In general, you must adapt all cut-offs to your data - metabolically active cells might have higher mitochondrial RNA in general, and you don’t want to lose a cell population because of a cut-off.</li>
</ul>
</li>
</ol>
</blockquote>
</blockquote>
<h2 id="applying-the-thresholds">Applying the Thresholds</h2>
<p>It’s now time to apply these thresholds to our data! First, a reminder of how many cells and genes are in your object: <code class="language-plaintext highlighter-rouge">31178 cells</code> and <code class="language-plaintext highlighter-rouge">35734 genes</code>. Let’s see how that changes each time!</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>log1p_n_genes_by_counts</strong> &gt; <code style="color: inherit">5.7</code></li>
<li><strong>log1p_total_counts</strong> &gt; <code style="color: inherit">6.3</code></li>
<li><strong>pct_counts_mito</strong> &lt; <code style="color: inherit">4.5%</code></li>
</ul>
</li>
<li>Everyone else: Choose your own thresholds and compare results!</li>
</ul>
</blockquote>
<p>We will plot the raw data before applying any filters so that we can more clearly see the changes we will make.</p>


In [ ]:
# Raw
sc.pl.violin(
  adata,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-raw.png'
)

In [ ]:
genes_filtered_obj = adata[adata.obs['log1p_n_genes_by_counts'] >= 5.7]
genes_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_n_genes_by_counts'] <= 20.0]

# Violin - Filterbygenes
sc.pl.violin(
  genes_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbygenes.png'
)

In [ ]:
print(genes_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-1"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-6"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-6" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-7" style="max-width: 90%;"><img src="...
"" alt="Violinplot-filteronce. " width="2715" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-raw-filteredgenes.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 7</strong>:</span> Raw vs 1st filter - genes/cell</figcaption></figure>
<ol>
<li>The only part that seems to change is the <code style="color: inherit">log1p_n_genes_by_counts</code>.  You can see a flatter bottom to the violin plot - this is the lower threshold set. Ideally, this would create a beautiful violin plot because there would be a clear population of low-gene number cells. Sadly not the case here, but still a reasonable filter.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">17,040 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
counts_filtered_obj = genes_filtered_obj[genes_filtered_obj.obs['log1p_total_counts'] >=  6.3]
counts_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['log1p_total_counts'] <= 20.0]

# Violin - Filterbycounts
sc.pl.violin(
  counts_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbycounts.png'
)

In [ ]:
print(counts_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-2"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-7"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-7" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-8" style="max-width: 90%;"><img src="...
"" alt="Violinplot-filtertwice. " width="2745" height="986" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-filteredgenesxfilteredcounts.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 8</strong>:</span> 1st filter vs 2nd filter - counts/cell</figcaption></figure>
<ol>
<li>We will focus on the <code style="color: inherit">log1p_total_counts</code> as that shows the biggest change. Similar to above, the bottom of the violin shape has flattered due to the threshold.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,678 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>


In [ ]:
mito_filtered_obj = counts_filtered_obj[counts_filtered_obj.obs['pct_counts_mito'] >=  0]
mito_filtered_obj = mito_filtered_obj[mito_filtered_obj.obs['pct_counts_mito'] <= 4.5]

# Violin - Filterbymito
sc.pl.violin(
  mito_filtered_obj,
  keys=['log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_mito'],
  groupby='genotype',
  save='-Filterbymito.png'
)

In [ ]:
print(mito_filtered_obj)

<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-3"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<ol>
<li>Interpret the violin plot</li>
<li>How many genes &amp; cells do you have in your object now?</li>
</ol>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-8"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-8" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<figure id="figure-9" style="max-width: 90%;"><img src="...
AAAAAElFTkSuQmCC
"" alt="Violinplot-filtermito. " width="2670" height="958" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violin-mitofilter.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 9</strong>:</span> Violin plots after filtering genes, counts, and mito content/cell</figcaption></figure>
<ol>
<li>If we carefully check the axes, we can see that the <code style="color: inherit">pct_counts_mito</code> has shrunk.</li>
<li>In the printed AnnData information, you can see you now have <code style="color: inherit">8,605 cells x 35,734 genes</code>.</li>
</ol>
</details>
</blockquote>
<p>Here’s a quick overall summary for easy visualisation if you fancy it.</p>
<figure id="figure-10" style="max-width: 90%;"><img src="...
"" alt="Filtering summary. " width="6592" height="4644" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/violins2gether.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 10</strong>:</span> Filtering summary</figcaption></figure>
<p>Fantastic work! However, you’ve now removed a whole heap of cells, and since the captured genes are sporadic (i.e. a small percentage of the overall transcriptome per cell) this means there are a number of genes in your matrix that are currently not in any of the remaining cells. Genes that do not appear in any cell, or even in only 1 or 2 cells, will make some analytical tools break and overall will not be biologically informative. So let’s remove them! Note that <code class="language-plaintext highlighter-rouge">3</code> is not necessarily the best number, rather it is a fairly conservative threshold. You could go as high as 10 or more.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-1"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-1" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li><strong>min_cells</strong> = <code style="color: inherit">3</code></li>
<li>Everyone else: Choose your own thresholds and compare results! Note if you go less than 3 (or even remove this step entirely), future tools are likely to fail due to empty gene data.</li>
</ul>
</blockquote>


In [ ]:
filtered_obj = mito_filtered_obj.copy()

sc.pp.filter_genes(filtered_obj, min_cells=3)
sc.pp.filter_genes(filtered_obj, max_cells=1000000000)

print(filtered_obj)

<p>In practice, you’ll likely choose your thresholds then set up all these filters to run without checking plots in between each one. But it’s nice to see how they work!</p>
<p>Using the final <code class="language-plaintext highlighter-rouge">filtered_object</code>, we can summarise the results of our filtering:</p>
<table>
<thead>
<tr>
<th> </th>
<th>Description</th>
<th>Genes</th>
</tr>
</thead>
<tbody>
<tr>
<td>Raw</td>
<td>31178</td>
<td>35734</td>
</tr>
<tr>
<td>Filter genes/cell</td>
<td>17040</td>
<td>35734</td>
</tr>
<tr>
<td>Filter counts/cell</td>
<td>8678</td>
<td>35734</td>
</tr>
<tr>
<td>Filter mito/cell</td>
<td>8605</td>
<td>35734</td>
</tr>
<tr>
<td>Filter cells/gene</td>
<td>8605</td>
<td>15395</td>
</tr>
</tbody>
</table>
<p>{% icon congratulations %} Congratulations! You have filtered your object! Now it should be a lot easier to analyse.</p>
<h1 id="processing">Processing</h1>
<p>So currently, you have a matrix that is 8605 cells by 15395 genes. This is still quite big data. We have two issues here - firstly, you already know there are differences in how many transcripts and genes have been counted per cell. This technical variable can obscure biological differences. Secondly, we like to plot things on x/y plots, so for instance Gapdh could be on one axis, and Actin can be on another, and you plot cells on that 2-dimensional axis based on how many of each transcript they possess. While that would be fine, adding in a 3rd dimension (or, indeed, in this case, 15393 more dimensions), is a bit trickier! So our next steps are to transform our big data object into something that is easy to analyse and easy to visualise.</p>


In [ ]:
output_h5ad = filtered_obj.copy()
sc.pp.normalize_total(output_h5ad)

<p>Normalisation helps reduce the differences between gene and UMI counts by fitting total counts to 10,000 per cell. The inherent log-transform (by log(count+1)) aligns the gene expression level better with a normal distribution. This is fairly standard to prepare for any future dimensionality reduction.</p>
<p>Now we need to look at reducing our gene dimensions. We have loads of genes, but not all of them are different from cell to cell. For instance, housekeeping genes are defined as not changing much from cell to cell, so we could remove these from our data to simplify the dataset. We will flag genes that vary across the cells for future analysis.</p>


In [ ]:
output_h5ad = sc.pp.log1p(output_h5ad, copy=True)  # below function requires log scaled data
sc.pp.highly_variable_genes(output_h5ad)

<p>Next up, we’re going to scale our data so that all genes have the same variance and a zero mean. This is important to set up our data for further dimensionality reduction. It also helps negate sequencing depth differences between samples, since the gene levels across the cells become comparable. Note, that the differences from scaling etc. are not the values you have at the end - i.e. if your cell has average GAPDH levels, it will not appear as a ‘0’ when you calculate gene differences between clusters.</p>


In [ ]:
scaled_data = sc.pp.scale(output_h5ad, max_value=10.0, copy=True)

<p>{% icon congratulations %} Congratulations! You have processed your object!</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment</div>
<p>At this point, we might want to remove or regress out the effects of unwanted variation on our data. A common example of this is the cell cycle, which can affect which genes are expressed and how much material is present in our cells. If you’re interested in learning how to do this, then you can move over to the <a href="{% link topics/single-cell/tutorials/scrna-case_cell-cycle/tutorial.md %}">Removing the Effects of the Cell Cycle</a> tutorial now – then return here to complete your analysis.</p>
</blockquote>
<h1 id="preparing-coordinates">Preparing coordinates</h1>
<p>We still have too many dimensions. Transcript changes are not usually singular - which is to say, genes were in pathways and in groups. It would be easier to analyse our data if we could more easily group these changes.</p>
<h2 id="principal-components">Principal components</h2>
<p>Principal components are calculated from highly dimensional data to find the most spread in the dataset. So in our, <code class="language-plaintext highlighter-rouge">1982</code> highly variable gene dimensions, there will be one line (axis) that yields the most spread and variation across the cells. That will be our first principal component. We can calculate the first <code class="language-plaintext highlighter-rouge">x</code> principal components in our data to drastically reduce the number of dimensions.</p>
<blockquote class="comment" style="border: 2px solid #ffecc1; margin: 1em 0.2em">
<div class="box-title comment-title" id="comment-1982"><i class="far fa-comment-dots" aria-hidden="true" ></i> Comment: 1982???</div>
<p>Where did the <code style="color: inherit">1982</code> come from?</p>
<p>The quickest way to figure out how many highly variable genes you have, in my opinion, is to re-run <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with the added parameter <code class="language-plaintext highlighter-rouge">subset=True</code>, therefore: <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes(output_h5ad, subset=True)</code>. This subsetting removes any nonvariable genes.</p>
<p>Then you can <code class="language-plaintext highlighter-rouge">print(output_h5ad)</code> and you’ll see only 1982 genes. The following processing steps will use only the highly variable genes for their calculations, but depend on keeping all genes in the object. Thus, please use the original output of your <code class="language-plaintext highlighter-rouge">sc.pp.highly_variable_genes</code> function with far more than 1982 genes!, currently stored as <code class="language-plaintext highlighter-rouge">scaled_data</code>.</p>
</blockquote>
<blockquote class="warning" style="border: 2px solid #de8875; margin: 1em 0.2em">
<div class="box-title warning-title" id="warning-check-your-anndata-object"><i class="fas fa-exclamation-triangle" aria-hidden="true" ></i> Warning: Check your AnnData object!</div>
<p>Run <code class="language-plaintext highlighter-rouge">print(scaled_data)</code>
Your AnnData object should have far more than 1982 genes in it (if you followed our settings and tool versions, you’d have a matrix 8605 × 15395 (cells x genes). Make sure to use that AnnData object output from FindVariableGenes, rather than the 1982 from your testing in the section above labelled ‘1982’.</p>
</blockquote>


In [ ]:
pca_components = sc.tl.pca(scaled_data, n_comps=50, copy=True)

<p>Why 50 principal components you ask? Well, we’re pretty confident 50 is an over-estimate. Let’s visualise the variance of each principal component.</p>


In [ ]:
sc.pl.pca_variance_ratio(pca_components, n_pcs=50, save='-variance-ratio.png')

<figure id="figure-11" style="max-width: 90%;"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAYcAAAEbCAMAAAAGZLh0AAAAS1BMVEX////o
6OgbGxvDw8M2Njb39/d7e3syMjL+/v77+/thYWG1tbXQ0NBAQEDx8fGdnZ1I
SEiSkpKHh4dSUlIFBQVubm7c3NwrKyuoqKjkss8tAAAACXBIWXMAAA7zAAAO
8wEcU5k6AAAdf0lEQVR42u1dC2OjqhIGFAQE36///0vvN6Bp2u32bGwa95w7
dLdWQ4zhY57MDEL8W5uztz+tsH/+Pi2c4PbUZvPwu/0/tyswuE3vB8ko/WLY
fgANJx7hS0w6PyAjnHBa2IeAcPSPsXi6qNYPU5AVjxEQt38YUMzqpiGieGhU
QT6Sh++5OOhQ6aSJPoREM7Y8fj/QmoM63viNfW9VNIeOK4zP/S2L7G8Md/59
Gz6p98FsSEy4JCzsDY6b9MBb5AHEMEt0Pow5FhMnWjvilx+atpzWXoi68p1v
lRWDmspW2Lbzat3A/L1SHQY8qnHLaK1dt9pZqXIT9TStY/C9Feg8spw408KE
Oa8KUQddqIAh7YNclKDzTS2in+qlx2k3LxqdukJuUaYZP81BinoBYFKYGUQU
WzF0YfGeB/WElirKzRarS/ypG0QxYTrXKrEkWW1iwyg30yKncMgBCdCoTXVm
X3It0nUB8hmNEIXiUT3T+lL42YltnNRkAIHIQzmsUzltugczcpMkqgFAUalJ
qS3Jh2kBCkOFXoUYvEj0kCCamDGd8WBgfoP/SNUKjekMCBx+6YAhlWUvtjIN
+aLS4M5GiMNOAA7oiF6qFoaETOzpnwiKcTjV4rhCBQKbqaeEgyWiWMCI2qkV
bUVDHkTpF1HncS+aGw610hISRPQRJAJ6aNXS+O7faTY5cbGy1ysD262HPuTB
l0opHOhBmEnNsbUt1B+NIZezmrwWdZyUDyQXkpjQ0KJ8WQsZp27xg9CDUvPy
FwysfnQ8rbjSZZw/Vt583W4/cW+vk8SWn/lUl+NdMr3UZEvE2r/Mhf/HwP1V
ypPbH0nr/Ie7PaJ7t1iXBr05nt4l36zL7xJ/BTk8Zte7/4ztaf8+hfyhZ7LX
j5/9OHX0P9OsvX/4v8+hpN2DT9Uc30v+y5av//KVuMendwJAN3Xzd9H011/P
/uVsCZLrwSeSKrV1qkr8vL6V6UPL/ViVK/3Lz1Ku5f5Mb4+29zpeKo9L5d3v
6pIv8q7BL2MfIVR4iuO0be3WrmxPPrFVj7Ox1UjSmnrNo3cdDuQdkwdH4+G7
DgchlAkwnwrDg3ctDmtsSKJUHCF6KQ5N7Gj83ciDdy09lL6vRTDMly7GoYLf
eZo3HrurcYhbbTwH617Pl+DIHxuO+blaby3NYJTpGx69S3GIW1HXdVHz4F2M
g/6PrQj9W+24OwHNcvpiHPY0DiaJK+0HwTlmf4PeyqPGODAOjAPjwDgwDowD
48A4MA6MA+PAODAOjAPjwI1xYBy4MQ6MA7dHcLhf7/wsl8Z+ggMvyv0APbTl
OtNRjmWHVPyo6KwuV3+LC6j+BbmX/3ocmjI0sQA9tLPcUAskxbE2vm789p4v
3YbecRrED+AQImrlUKpPh9oHlLRVo7JLoGou8+0tlgnix3EgGtgw9GJFnSiF
dPseF5bOimIvBtI08ajhnFL37d9WYuC/gUM9JiwQPYmwSaqH085gTmBFRUyv
m85Pd4WZ32opcHsqDsXohBmaG18CKFSpBXiYPOsbGd+qCB81Irk9XW9Vi10D
pYPOiTZAD6SqFnqsj1jKXW9djFmE7Tld8Wf01npVfdNvVL4oomwRyqqhml1R
KSM/2HFdUY+96BiFH7Kn7V5Ew1r7waZz73CIVsjeV28VN7j9mF9D3xczd+/o
wXkoVLLOpe1YRDwfh6N2lP1dJZEyv3qUFWZl6QV+PvtbP581AS8WhuXDi/yt
v1QiL+/D7yMP4cv83vZ9RalDX3KMw9+w/jDMQVPlTW7X4qApm7227Oa7Fgft
2xZFstntfTU9lFBZO6s5PetiHNbBGDUM7F+6GAdkshcpm92xgLg4XkMfm+jx
OF6Ig82FzpkUrqYHe3NIcT47x5ExDtwYB8aBcWAcGAfGgXFgHBgHxoFxYBwY
B8aBcWAcuDEOjAM3xuGvx+F+n0snPgvGqHjUXpC3m9bWjihu+1kcPodSvoIe
+qrc89irEvuXD6pC4rRJeexHHgoHyvw8DjIuC6XC2dbI2otQyaWUcpWiO/LY
NcuHV+Sxl5THTtynFlrpAXHEZVhUo8dCf5HHzu3JOLSeUnVpF1FNeex+k8K3
wigV90S4xnU8ai+oJzDn/0ItlMc+gzTGIngpfZ8Edx+7iUftBXnsVF+DkhrG
gtLXiS+NNSVS98jAskmBYvnwAhz0Gty6ZAbVehdWICPD2lhwJ7bjXll/qVKD
NSjq4BXq/wizVmBIplq9/JA/ze3n/RqHId2kf/p9mhzj8Coc3vszGvYvXYKD
JQz0R6+GZRxeioM++NBvUGAcXigf7H0ph0Qfb38zDq/C4fd+C8f0wOtAjAPj
wDgwDowD48A4MA6MA+PAODAO3BgHxoEb48A4cGMcGAdujMP/Cw7BtIUfFx7J
a3FwXbGVIXBOxNX0gHLr0TGvuhqHBvvV4IdjwK+mB32rXMntShxmyk8Jnkfy
YhwiBeNr5kuX48A2xc/Ft/4TDnc51f0c7JKThbg9N4/d3gZa//NbJLbj8AWn
Lj59P3Yq2K0TEC4BcTv99C15Ow6G4Qf2Y68MHRdfYavXPY9djgp5vJ/LB4lt
wlltfXoeexdkR/ux92apkR9XNkiOE+D/Un8e712aGdtxGB7Jp+JgAy7MlJI4
pn2PW+SLdiG8S12v3LtE31AvW1HwSD55H3BPW00jFasK2AO5oSxRvy2rL/2S
xbZu4iepKywgnp3H7m/7sTvksRsIB18XUxDzkDZf76uPeey83e6P5LF3VF/D
7vuxpzz2WNB+7EXH60AvxMGq0JSB7DOqN5Py2FctsCM4RLFlHF67H7ujPPZY
Van+0lpTtYd1lEwPF/iXmvc+Dn0nCBiHF+Xtfn7imB6uqetAKBxl+tLuZCwf
XomDzlRg88jjp3mjBc04vFg+3GoK2Pud4hIgjAPHkTEOjAPjwDgwDowD48A4
MA6MA+PAODAOjAPjwI1xYBy4MQ6MAzfGgXHgdgUOHF15OQ5haULhOdz4ahzU
OhozDTygV9NDb4pmZsZ0vXwIm195PK/GgXLpZODxvBoHQyK6NsyYnhV3j5lt
T+AQBW+J/OT8aevutwn61TT4HAeKBteMw9NwOAa8sR8B0F/ypT40heGyAk/M
Y09jT9O7+VBkYA8E/1xfGtQ01ywenkYPG/a1FCmRuqR9LYcV+UCghMo3X9JD
126Gi/88Dwckxy00/g7Ju8gbDavDluxgO+P4tXygtC0u/vPkfY97t+eLuj7l
i4ow1uPXfEm1KCswc1mBp+ax5/xpS/uAp/xp4Ysifk0PRV1Q4wF95n7sCQcU
1FAy1xOoZ1GPWTz8msd+qFUso5+JA1ViNVRfA/VlFqWJL3WFmVY1DTJXRPxd
Dbjm0HI1lxh4gpxeF1mCvdhdTleNXJvEr27505X+7d0K0czdzMWNn1F/aVVG
9Bvlsaf92Ku1PeSGyxP9ixTTiDI1smC96Rl2XB5ve3AavefvNv+4zytejLpz
6Z4sLJ7jb7V3leD0B69H+VsYmi522sqO5cNz/EvuXUXKj4BUv98VOQMgeVh/
qG6ovftPb3G/oSJDRfzqwfG4Xhu/VHGA08U45MoDydij4vcspy+lh35e5MKr
EBfjYEWzTRO5pLhdSg/OD7XxvxXj3F4gH1K5oDLZ1JqBuI4ekmRGYeNBDQNL
6Qv5EqlKaQ2i5lWIvyD/QYv7mAJu1+Wh2GM9goG4Nh/IbSJ0na95dC/Oy4qi
W8TCyxBX4zAuCKORHGR5NQ7BRFUyX7o+XxTUYHmvmstx8JSQUqx9w1hcikOV
rIep72IvGYrLcGg8rY86bN4h227kVdIrcEhxfcaEJsyruy1Ws8Pp5fRAeyDr
dlK+PrZvYrP6EnpA67AMMbs9dZFhuEo+UASs9vt+sCtTxDU4gB0hEaKfBpWG
vxOsMV2lt6ZFiGI22BF5HlhOfweHuzRRd7L+Eu2IPNeJQuS7LYbeBea7G73Y
30Z2PtSstf8RHNxbfPG77VD2P/4QB+m3tvcUKT7Xc1Xn4V78GCME+DFWdh/u
X5KEv0FC9j9CDzrvEeTyfHT6bD0y7IisR9FEGVTYP6fpAm4Xxl/H2/2f19Wq
PkvrcfsW4O5tw6y7kz/FYR0MUhf70jbIXCnzTSq6fejE235cmneL/QwHLd4C
u+1ddYe7EPA/xWFLuYv1EL0ZY58T44s4jp3f0r10K+1td67m/7y63Cf5QGvK
vm1Gha2Pkcde9mAkqjS3EXo0jBgDLXUYSjW2yNhqGgjtNjVlWph6fZjb2qdI
8Z0Yv9Os/Y/sx14ukvKxsK+rrik/ToYomwKrnMXJepV+ceQEL2TwZFzTOBnV
bm27TUXrN1t2RV3VhQ+NLHz4bqSHc/a/s+8x6fwpjz3li8Y0OPEsDl1iPFOK
A48Y/rZv15z0CIdsM8+xTLecovfzNBdWmmgCIg185cMpHOx/I4/dinZOeez6
yGOnwOGwNmdxyJF+3VjWIIRt29pNiX0ZGxO/neAlb+AL8VBnR4QYxBokGAl9
qdqEVzF6xH+0WGz1gwtj2a0HQn4cMT3mojNFGfsaAgkf4vYkP3tLJbZfKWL2
Tmne9RGd7tF87gnQd3ZQZqXuS7lmf3Ou/2A/dko2DFhB0D7VE4DhVbb59T5+
nsf++2aG4BbfS4cxlYYGtunzs5jDpqNTBHgMKPDXSdBKg1ULZP9KNZCAl2NY
lJTYGDuMhY8LuodohtmYYVqa0YhIqrGWqpsHcDyIM192Rmyj8n0cCvoMLIPU
sFtmhT5S+thFsyTVY5Dw0eNJFr8VHrmuYQFvROa46RAvjQ/rwtgn332IHjNg
m0vMBa19xN0CPbfe6oHuMNypfs1bChWh6Zo3XTSbS/ZXK/XTPHZKYBgpj71p
Tcpol92tImjjuseEtEVFIF/MC56m7rNyakk89G0K6Gg2mSYhfSYK/JVjKWHt
Ba8QaUArSQC+Ehq+KkVWoIiIyIF9Xqo6QBmbcLfWKzyR16IaYL2PurNjoeWk
gecyrrVf/SJHwCjFJEFacsQ2507UZY9nqNdYbGMrujK0ZRHwkeMwzaWY26ZW
CP7RsixRQaFfxOowF9wUAE7h00bpherbrW/VWGy+FxV0Du8hBBtCssCM63C1
GEu/+VbSdx/nYqxqiURo3G+BObveR1L8ordWQe/7sS+0H3ulUXJG+Hm3KNJb
HmLAcmzJrC7TNFlNgRkSjOqTnG5rvFaausdpmWbNnsJCDyBBCPmRTOwwGOUY
MTdN4auxHIOhsmkOoGBGRp/td9G0vhJloM9Syxig55HPF+NWCbnqBlA0wKkD
8LqZSGPrywhQzQw+SDXt1ADMvOgaepui4B8XK/qEMkYrYftgOsjRjan4Grhm
22+bQnZs6GdV11UI44T4uXkyFZi5LFAWI2AuqL4bTehkscqmjKBqH2bU8LHQ
fdwXemu5DgJa5DJWENXg6FUr6mldy3pXZPRD8sGJ6MisHols3Zjeq8uZvojF
hAIaak2HqQaFJMkBW6KV6QnnNz9TA/xkc0f1d24vmR9MEheQYpy9dLoynVIz
Zd8TAQKqNpa+i7GwbQuu4secbWlG+qRejRhMIjdpgGQcaVJMJk7o3yV+I4fS
m4ip4GMr5g1TAj9JeBmSVVj4Sh7+coFzU8+yowp5WjUpjAuJCHIbJ5xLUXZE
7ZoOGtPkS79Gs+99TIqH+2ST9vIxR7Yis9qUAzSh2US6s/bZWvQ6KbEmjSNm
F2ikakEbda92xpU+vs+WXuaMZiF27IYFYgdAGfB53Qbcwwoz0yWzy1tp30Sh
pu9y2KSQO6CBVIYwfbDLX7PZq7A1IfcjTK3MN9N7thMkt8xeHytvDCLf11NX
WnhZjC8tuE7j9OpXBRuJlENnh8oPXRz6kfgVyCPG8iu+9F5/1++lvTtTlBIm
dYBhDSkBkYNBdMH4JD2lF3e7jKfYAg0EaiIL4r2gEeJe/UqXiGB68IEq1jUY
ekl8fbOqLroNMqWoy6JRRY3yOF1IktebvhB6oPmJuwYz4Dv7YZaE2kysfEMH
P4INbaEbl8XjTnM7DEvng5z7MKLDVoyjLKg3OswtrtX0e5QBP4Wv5RKWjn7J
0IXbttAQBW/+ZRAZobRLaiJoSXasvdtG+o/WH+x79cqdWbLIS6U9lkqpzNk0
YRqkW2Dg0sLpTIdiL6GV4BDDbmAQD99W1deAg2TK1q6YBY3xxNCdN8kCMSXN
DD/TiYEsSUCBX0OalF0GqktnQC0mCGsVAk5UscQybGuQsSw2ktM1Xlm61GHB
yYrabLAy13xtxZvwSrGUqiNbR1XeezWqmA7LsshuIVTA/yWggeW7ALAR/wko
6GJy6RbqBvAKh4IwXr5+/4eOPCVa+L4e/HGLbEZAEhLWCZRifmdNz2nqGDoI
W6anHkaaee00LulA11qV5GY7JfVCEeOfCRRB8pfEMNRYumam9Mouk6f0GycO
TByHBnbOfg1/kWGZ2Az1pvNdkKffpSSh0IHNFjhdzIyTjIYna9RMczZKp3wt
v7SfTAnCOUKQ67pzL8dBGZIS/UpwQK9Bfi+0Jpdc4WkURV5FjXMBhTt4Yi2w
A+6toFvNTH1XyvoONZflQjAZqIST6ogVtCph2KsMFBkoooHGAknRQUBLOlg6
oJvTLibxKWGogItFyDLqBzolg0ZSN5hXQAJfJQwjDSUMnySgG6+BTQC/Tdj4
gjjkaDNQxC6td7mDjZp8/+Xr6aEIJCXq0tDK9SC3Sc1tDzERBtPjySRNYDDB
WB40kta5E434GcquqGOd2BdxMV3MdLC4BpMon4giaXR18iUmHePmLNZ69zbe
wxjuNI/j2nJ/bRF3y2G7RF72VxzZOskkTd4eS5XNMdw0mEU/w83vlp6sn0Cm
qbAAqsIlM68kFgd8125EJxmbq/alSSK7qMehoPxeo0hY6EGpsR5mWF9zn0hV
Z/3WqlxMKAd+ZDO+GtMIEZ9ojmtl1hymNNKVSRRFqDXFQDuDoL4veXNtiAX0
DpwksQTBDt1onrEw5eiASRDKQC7J1FuHCicOj4lXIKzprjLdVYY5dVhWdNA0
MZpcnkpnUBxAIdGsacmLTpK5QieNDAlB9MuRv+6i/YH0oSWQdj2j+F87wEDs
th4HcA8CxRBxJBqxwSQBsptNcHglSiErDMZU1EnS5zCdrMnvnC2bKFNS9oi7
o+OaiK3JqJGkx5lKait12A+EZK4vlcdkSspi7NIdiECb4w7Vrmbo43ZkoRYy
k65+Q7IkkqFwCTpURT6pofwW5EsTujZX79O0msFMvcpKMFi0QwZRMrszjWyk
wE6mJlBAzUNfOOk7MGD4qUxfQwYgP7hAMMjolwYd+iFo6oBLvu+SjpcGyWVQ
pB+T5y5JCdjTeocVM5zcJARr9hGsCdwxdWjypaojOGTXNUmokFGnswVHEoZm
SbqDnZJw6vKy485dM2rxRq3goV1+BVC794P/chzSlKuTTZEtPJNE+KBgRsDo
Bhyk34791kMcIj5z3FBOKJHKkH7LVikygMgeKSBn6BqBB+Ok3ykKUseTG64e
hh6yBSJoMBJmS5U8jgNsGL3MVV/YjKcr0A/dyd0CPPE0OFnwSLg0t2QcUj+6
q4mzBO44W5JzhuQaaLfQOCGkZReJ2h2Ntk72M0DsYnL8JtwTC4WF2HUfLbFX
4mB/DXPK+db4qTMaJtqsO+YsO4rPxATtqVrEXrkaNiCV6wVQxgifOoixBekc
JorIJgp+m40cDgmofOjTb3d0SHjiGtyDOmPYA/e33ulSetO2X3NbftP9HVzq
NyQggTs8gTj4BSDHZLSma8F39AooF35kXINFCb/v8D0cfmKhRYccfLbOA+ay
WQ20j6TYZv1ddOTmtSAYsJgGh0jkA7+VoOsk3McGmmA66eseK06w+cYUaQtu
Z+naKMo2AYWXcK3aNlzSdJL7tR11IOumS7jHlt5DvdO11C/2tw7jfh96ZRhF
Il3I34xQQ3gd1LrUOBRNln+argWb5kdxZyM/uqWPdj+12cbuEygC4CCtKmz1
O8Y1DYlgsiFyXLs7MflkAPOWM6VgLD6Zj+Q+AXDLTJ5wuoADPLnoIGEbjPLo
t/iGXvHJsIPPlfzms0690ZcOM/0B88fRXJhlvk/yr3sdASvdLqM2Ur0X7UHB
2LrKE17O0yuW+O6Q+rWD/7iPyYPzW71ssXA7GBf5qHpYIFsgwZIOBVkkfX71
rUNdbBWWiAiUfkdoUDPV/EAID0DKSA5m74BXjn57h0EltFOHIb/S3w7zNORr
VEZkRkzQW4eVUAOvaAgbomDyn8P31wG1QNdigpCu0cSgfv6+nOSJLa7WJ01/
+8dSxb2TLvqXsoFHnoW96QChzd7F3cmYr/UQQ+kK4dvni0WB8B7wxHrvkImR
XhoIVsJ9oz/2a6njgXuSa+km0DkO1ChmCyKcEJ4GmgHgsQQUyLm/TYKdgnt3
e+ry8Yih+KqyoPqeZdk/661vTg73i4KgP/2u+r4Uxbvr+t1j3CraHnduPhio
IS0RglYLgm0HioArEu4JsjQfMth0eMdkHoxSwTpwVVXlFMuyKtfqZCvjVK64
we28/PjHioLK6JG64Liie/40eh/1wx1wA5zRP+qfe6/4KWN1uzfelTukK3jb
WsUq3zNOHV65+xJlmT4q/5X7V2W+A33Sun/l9ODppFK4V3p/qdIDUi969Py0
kR4mlnSOE+qBj4/5Hvh6ePNtHORjOLijzLH/ZnKJe+QG7tMIGe/cn3X9hQia
/dqov+hsP7n2y92j+4OP/4qA93FoHg1ltM+SEuoPbQ338cpNIqj3z/15QMqX
916/GUCu3+7g3sKAf0XE/cLHfhmHs3n/5XcD76pzgaz2SYa9PWk93edtuO/c
QX/vWxzj0H5XBrenP986u9/hu0kS3/kSzXe+RdqkLyMxXBla7b4zfn/MgP5x
PupvfYFGvA/te9wkPvYXeH2zb5NBn9k55WPainyacnzinY04YjvOjMRR1vM7
+8dsORD/XJMzLQiEruzCydB6xNmtCJ2UCJ08u8cvcgm6GWG7cTy9fQtCk7Dg
tHZn54ImRdZqaLLy1IyG0Y6I0vF8WcqwQd0zbWhP7c9LBrbEukMsKBLIPGKb
f6hNFAuEtJqTYsYhqqTDcn8YzsgICkt3K02BuZf92TkdSkfR+afFi1S0IRdm
41kBS5K6DBWW7qvz3FXGGssG9dn6dcG33mHVro4nBa1NK0kKEdYnXXY27TL3
jYoNck16s1Xf0HUQ9zsdi16nxOSqPC2syrOqpy9CxA0Q+Hp2Pk4Rmy+pXJrn
zCA4is5vZ3tWYUOaEXCw5123FDcZN/EtHDCAXY35sJzEYTOu6Iiw5WmTFrkI
VU3WqDo5kgFrQWY4K550enQkWIT1ND00Hjx1pZAiexIIzAdPgWgn+RJFmWBl
Z2ds+qTq60wLvhTUSd1ZVnI5K6cpHqwo8QR9yq549N0pNhkwzBSQf5PTJ5Qu
Wr7ZSE6f365iw1rPJof2rO5LkSK1aWU76JPcGXrrfFZvJr11UvXSrfGs0tmE
SZWIQY1qDB+Dof+wFfB2zk6e11ttSpwKsUz5FGfuUIyrGixU8NPj8B1Dmmw4
+2G54CGF79im/Jae9o3MRPe9PThdNiSdPm1D2qx12yt8E0dy4NlSJgkKfaQV
6tPPYr/zPdwzTPLkvr/EsfS+bsqJKWTvlkqf4Jtxl36b9/6ylz/+tysG/AUp
0vaOHLje2i/MwX4xYjxcr+Z3R+0anRIcuV2wpHHTPfRAhqbcCqe/v1jB7QGC
uBViyG7kwdDV5Rb/wrsAv6YFNXbjGimC1ceqoLIIvW9MLVc/rgjyHWOnuMj1
z4tpZHNqpKNjZWiqdT/roae0U8Rvq1q23ppeholx+PlGXnxkJk6bQEJiEYVZ
KWB4zKcI5oY/bGLe9PNiGj5orElghczSElO0g4cr06JykaJ6dLTuJJgvvaAR
DiugaNPAl5APRUWwUJ0ZlNFBRd+a+dIr+FJnGyT2x5ZSn1HgB3prXUok8dAp
8s07FBJivvQKgjgsOXkXmu1usTVY8uGd4l9mROi73HR9X+8txJT7xu3nQbD/
4IzkrTZ/HgQnPhRFdH+Rj/j/yrt07w93v3E/MRavWnP9crjZ2ceNGzduD7b/
AU5CTGtLa3gWAAAAAElFTkSuQmCC
"" alt="Variance ratio. " width="391" height="283" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/pca-variance.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 11</strong>:</span> Variance ratio</figcaption></figure>
<p>We can see that there is really not much variation explained past component 19. So we might save ourselves a great deal of time and muddied data by focusing on the top <code class="language-plaintext highlighter-rouge">20</code> PCs.</p>
<h2 id="neighborhood-graph">Neighborhood graph</h2>
<p>We’re still looking at around 20 dimensions at this point. We need to identify how similar a cell is to another cell, across every cell across these dimensions. For this, we will use the k-nearest neighbor (kNN) graph, to identify which cells are close together and which are not. The kNN graph plots connections between cells if their distance (when plotted in this 20 dimensional space!) is amonst the k-th smallest distances from that cell to other cells. This will be crucial for identifying clusters, and is necessary for plotting a UMAP. From <a href="https://github.com/lmcinnes/umap">UMAP developers</a>: “Larger neighbor values will result in more global structure being preserved at the loss of detailed local structure. In general this parameter should often be in the range 5 to 50, with a choice of 10 to 15 being a sensible default”.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-2"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-2" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Number of PCs to use</strong> = <code style="color: inherit">20</code></li>
<li><strong>Maximum number of neighbours used</strong> = <code style="color: inherit">15</code></li>
</ul>
</li>
<li>Everyone else: Use the PC variance plot to pick your own PC number, and choose your own neighbour maximum as well!</li>
</ul>
</blockquote>


In [ ]:
neighbours = sc.pp.neighbors(pca_components, n_neighbors=15, use_rep='X_pca', n_pcs=20, copy=True)

<h2 id="dimensionality-reduction-for-visualisation">Dimensionality reduction for visualisation</h2>
<p>Two major visualisations for this data are tSNE and UMAP. We must calculate the coordinates for both prior to visualisation. For tSNE, the parameter <a href="https://www.nature.com/articles/s41467-019-13056-x">perplexity</a> can be changed to best represent the data, while for UMAP the main change would be to change the kNN graph above itself, by changing the <b>neighbours.</b></p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-3"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-3" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you are working in a group, you can now divide up a decision here with one <em>control</em> and the rest varied numbers so that you can compare results throughout the tutorials.</p>
<ul>
<li>Control
<ul>
<li><strong>Perplexity</strong> = <code style="color: inherit">30</code></li>
</ul>
</li>
<li>Everyone else: Choose your own perplexity, between 5 and 50!</li>
</ul>
</blockquote>


In [ ]:
tsne_components = sc.tl.tsne(neighbours, use_rep='X_pca', perplexity=30, copy=True)

In [ ]:
umap_components = sc.tl.umap(tsne_components, copy=True)

<p>{% icon congratulations %} Congratulations! You have prepared your object and created neighborhood coordinates. We can now use those to call some clusters!</p>
<h1 id="cell-clusters--gene-markers">Cell clusters &amp; gene markers</h1>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-4"><i class="far fa-question-circle" aria-hidden="true" ></i> Question</div>
<p>Let’s take a step back here. What is it, exactly, that you are trying to get from your data? What do you want to visualise, and what information do you need from your data to gain insight?</p>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-9"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-9" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Really we need two things - firstly, we need to make sure our experiment was set up well. This is to say, our biological replicates should overlap and our variables should, ideally, show some difference. Secondly, we want insight - we want to know which cell types are in our data, which genes drive those cell types, and in this case, how they might be affected by our biological variable of growth restriction. How does this affect the developing cells, and what genes drive this? So let’s add in information about cell clusters and gene markers!</p>
</details>
</blockquote>
<p>Finally, let’s identify clusters! Unfortunately, it’s not as majestic as biologists often think - the maths doesn’t necessarily identify true cell clusters. Every algorithm for identifying cell clusters falls short of a biologist knowing their data, knowing what cells should be there, and proving it in the lab. Sigh. So, we’re going to make the best of it as a starting point and see what happens! We will define clusters from the kNN graph, based on how many connections cells have with one another. Roughly, this will depend on a resolution parameter for how granular you want to be.</p>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-decision-time-4"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-decision-time-4" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Decision-time!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Oh yes, yet another decision! Single cell analysis is sadly not straight forward.</p>
<ul>
<li>Control
<ul>
<li><strong>Resolution, high value for more and smaller clusters</strong> = <code style="color: inherit">0.6</code></li>
</ul>
</li>
<li>Everyone else: Pick your own number. If it helps, this sample should have a lot of very similar cells in it. It contains developing T-cells, so you aren’t expecting massive differences between cells, like you would in, say, an entire embryo, with all sorts of unrelated cell types.</li>
</ul>
</blockquote>


In [ ]:
# Find Clusters
clusters = sc.tl.louvain(umap_components, resolution=0.6, copy=True)

<p>Nearly plotting time! But one final piece is to add in SOME gene information. Let’s focus on genes driving the clusters.</p>
<h1 id="findmarkers">FindMarkers</h1>


In [ ]:
markers_cluster = sc.tl.rank_genes_groups(clusters, groupby="louvain", method='t-test_overestim_var', n_genes=50, copy=True)

<p>But we are also interested in differences across genotype, so let’s also check that (note that in this case, it’s turning it almost into bulk RNA-seq, because you’re comparing all cells of a certain genotype against all cells of the other)</p>


In [ ]:
markers_genotype = sc.tl.rank_genes_groups(markers_cluster, groupby="genotype", method='t-test_overestim_var', n_genes=50, copy=True)

<p><strong>Note:</strong> The function <code class="language-plaintext highlighter-rouge">rank_genes_groups</code> does not return a DataFrame that we can use but instead metadata about the marker table, so first we need to construct the marker table using this generated metadata. This is done using the following function, however it’s not too important to understand what this code does!</p>


In [ ]:
def generate_marker_table(adata):
    # extract marker table metadata
    res = adata.uns['rank_genes_groups']

    # generate DataFrame from metadata
    res_df = pd.DataFrame({
                "genes": pd.DataFrame(res["names"]).stack(),
                "scores": pd.DataFrame(res["scores"]).stack(),
                "logfoldchanges": pd.DataFrame(res["logfoldchanges"]).stack(),
                "pvals": pd.DataFrame(res["pvals"]).stack(),
                "pvals_adj": pd.DataFrame(res["pvals_adj"]).stack(),
            })

    # convert row names to columns
    res_df.index.name = 'newhead'
    res_df.reset_index(inplace=True)

    # rename generic column names
    res_df = res_df.rename(columns={'level_0': 'rank', 'level_1':'cluster'})

    # reorder columns
    res_df = res_df.reindex(columns=['cluster', 'rank', 'genes', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj'])

    # insert ref column
    res_df.insert(2, 'ref', 'rest')

    return res_df

<p>Now we can generate our marker tables!</p>


In [ ]:
# Generate marker tables
cluster_marker_table = generate_marker_table(markers_cluster)
genotype_marker_table = generate_marker_table(markers_genotype)

display(cluster_marker_table.head(4))
display(genotype_marker_table.head(4))

<p>Now, there’s a small problem here, which is that if you inspect the output marker tables (above tables), you won’t see gene names, you’ll see Ensembl IDs. While this is a more bioinformatically accurate way of doing this (not every ID has a gene name!), we might want to look at more well-recognised gene names, so let’s pop some of that information in!</p>


In [ ]:
# Join two datasets

cluster_joined = pd.merge(cluster_marker_table, markers_cluster.var, left_on='genes', right_on='ID')
genotype_joined = pd.merge(genotype_marker_table, markers_genotype.var, left_on='genes', right_on='ID')

display(cluster_joined.head(5))
display(genotype_joined.head(5))

In [ ]:
# Cut columns from tables

cluster_markers_named = cluster_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]
genotype_markers_named = genotype_joined[['cluster', 'ref', 'rank', 'genes', 'Symbol', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj']]

display(cluster_markers_named.head(5))
display(genotype_markers_named.head(5))

<p>Well done! It’s time for the best bit, the plotting!</p>
<h1 id="plotting">Plotting!</h1>
<p>It’s time! Let’s plot it all! But first, let’s pick some marker genes from the <code class="language-plaintext highlighter-rouge">markers_cluster</code> list that you made as well. I’ll be honest, in practice, you’d now be spending a lot of time looking up what each gene does (thank you google!). There are burgeoning automated-annotation tools, however, so long as you have a good reference (a well annotated dataset that you’ll use as the ideal). In the mean time, let’s do this the old-fashioned way, and just copy a bunch of the markers in the original paper.</p>


In [ ]:
# PCA
sc.pl.embedding(
    markers_cluster,
    basis='pca',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# TSNE
sc.pl.embedding(
    markers_cluster,
    basis='tsne',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

In [ ]:
# UMAP
sc.pl.embedding(
    markers_cluster,
    basis='umap',
    color=['louvain','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='.png'
)

<p>{% icon congratulations %} Congratulations! You now have plots galore!</p>
<h1 id="insights-into-the-beyond">Insights into the beyond</h1>
<p>Now it’s the fun bit! We can see where genes are expressed, and start considering and interpreting the biology of it. At this point, it’s really about what information you want to get from your data - the following is only the tip of the iceberg. However, a brief exploration is good, because it may help give you ideas going forward with for your own data. Let us start interrogating our data!</p>
<h2 id="biological-interpretation">Biological Interpretation</h2>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-appearance-is-everything"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Appearance is everything</div>
<p>Which visualisation is the most useful for getting an overview of our data, <em>pca</em>, <em>tsne</em>, or <em>umap</em>?</p>
<figure id="figure-12" style="max-width: 90%;"><img src="...
AAAASUVORK5CYII=
"" alt="PCA-tSNE-UMAP. " width="3304" height="1234" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/louvain_clustering.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 12</strong>:</span> Louvain clustering by dimension reduction</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-10"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-10" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>You can see why a PCA is generally not enough to see clusters in samples - keep in mind, you’re only seeing components 1 and 2! - and therefore why the tSNE and UMAP visualisation dimensionality reductions are so useful. But there is not necessarily a clear winner between tSNE and UMAP, but I think UMAP is slightly clearer with its clusters, so we’ll stick with that for the rest of the analysis.</p>
</details>
</blockquote>
<p>Note that the cluster numbering is based on size alone - clusters 0 and 1 are not necessarily related, they are just the clusters containing the most cells. It would be nice to know what exactly these cells are. This analysis (googling all of the marker genes, both checking where the ones you know are as well as going through the marker tables you generated!) is a fun task for any individual experiment, so we’re going to speed past that and nab the assessment from the original paper!</p>
<table>
<thead>
<tr>
<th>Clusters</th>
<th>Marker</th>
<th>Cell type</th>
</tr>
</thead>
<tbody>
<tr>
<td>4</td>
<td>Il2ra</td>
<td>Double negative (early T-cell)</td>
</tr>
<tr>
<td>0,1,2,6</td>
<td>Cd8b1, Cd8a, Cd4</td>
<td>Double positive (middle T-cell)</td>
</tr>
<tr>
<td>5</td>
<td>Cd8b1, Cd8a, Cd4 - high</td>
<td>Double positive (late middle T-cell)</td>
</tr>
<tr>
<td>3</td>
<td>Itm2a</td>
<td>Mature T-cell</td>
</tr>
<tr>
<td>7</td>
<td>Hba-a1</td>
<td>RBCs (as impurity here)</td>
</tr>
<tr>
<td>-</td>
<td>Aif1</td>
<td>Macrophages</td>
</tr>
</tbody>
</table>
<p>The authors weren’t interested in further annotation of the DP cells, so neither are we. Sometimes that just happens. The maths tries to call similar (ish) sized clusters, whether it is biologically relevant or not. Or, the question being asked doesn’t really require such granularity of clusters.</p>
<figure id="figure-13" style="max-width: 90%;"><img src="...
wYJF8OL/AVMXzax4wTdwAAAAAElFTkSuQmCC
"" alt="Known marker gene locations. " width="1867" height="851" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/known_marker_gene_locations.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 13</strong>:</span> Known marker genes locations</figcaption></figure>
<blockquote class="details" style="border: 2px solid #ddd; margin: 1em 0.2em">
<div class="box-title details-title" id="details-working-in-a-group-important"><button class="gtn-boxify-button details" type="button" aria-controls="details-working-in-a-group-important" aria-expanded="true"><i class="fas fa-info-circle" aria-hidden="true" ></i> <span>Details: Working in a group? Important!</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>If you have deviated from any of the original parameters in this tutorial, you will likely have a different number of clusters. You will, therefore, need to change the <code class="language-plaintext highlighter-rouge">categories</code> parameter in <code class="language-plaintext highlighter-rouge">rename_categories</code> accordingly. Best of luck!</p>
</blockquote>
<h2 id="annotating-clusters">Annotating Clusters</h2>
<p>As mentioned at the beginning of the tutorial, you might not get outputs identical to a tutorial if you are running it in a programming environment, but the outputs should still be pretty close. However, you have to double check if the categories (ie. cell types) in the example below correspond to the identified cluster numbers based on gene expression. You might need to change the order of assigned cell types in the <em>categories</em> parameter to match the cluster numbers identified by <em>louvain</em>.</p>


In [ ]:
# Add meaningful names to each category
markers_cluster.rename_categories(key='louvain',
    categories=['DP-M4','DP-M3','T-mat','DN','DP-M2','DP-M1','DP-L','RBCs'])   # categories (cell types) correspond to the identified cluster numbers, ie. [0, 1, 2,...,7]

# Copy AnnData object
markers_cluster_copy = markers_cluster.copy()

# Rename 'louvain' column
markers_cluster_copy.obs = markers_cluster_copy.obs.rename(columns={'louvain': 'cell_type'})

# Scanpy - plot updated object
sc.pl.embedding(
    markers_cluster_copy,
    basis='umap',
    color=['cell_type','sex','batch','genotype','Il2ra','Cd8b1','Cd8a','Cd4','Itm2a','Aif1','Hba-a1','log1p_total_counts'],
    gene_symbols='Symbol',
    use_raw=False,
    save='-annotated.png'
)

<figure id="figure-14" style="max-width: 90%;"><img src="...
xvqV9f8AambrJWCX0V4AAAAASUVORK5CYII=
"" alt="Annotated Embeddings. " width="589" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/annotated_umap.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 14</strong>:</span> Our Annotated UMAP</figcaption></figure>
<p>Now that we know what we’re dealing with, let’s examine the effect of our variable, proper science!</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-genotype"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Genotype</div>
<p>Are there any differences in genotype? Or in biological terms, is there an impact of growth restriction on T-cell development in the thymus?</p>
<figure id="figure-15" style="max-width: 90%;"><img src="...
HBwcHBxrHP8fVgKajBU24VsAAAAASUVORK5CYII=
"" alt="Genotype Images. " width="1197" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/genotype_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 15</strong>:</span> Genotype differences</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-11"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-11" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We can see that DP-L, which seems to be extending away from the DP-M bunch, as well as the mature T-cells (or particularly the top half) are missing some knockout cells. Perhaps there is some sort of inhibition here? INTERESTING! What next? We might look further at the transcripts present in both those populations, and perhaps also look at the genotype marker table… So much to investigate! But before we set you off to explore to your heart’s delight, let’s also look at this a bit more technically.</p>
</details>
</blockquote>
<h2 id="technical-assessment">Technical Assessment</h2>
<p>Is our analysis real? Is it right? Well, we can assess that a little bit.</p>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-batch-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Batch effect</div>
<p>Is there a batch effect?</p>
<figure id="figure-16" style="max-width: 90%;"><img src="...
"" alt="Batch effect. " width="502" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/batch_effect.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 16</strong>:</span> Batch effect?</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-12"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-12" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>While some shifts are expected and nothing to be concerned about, DP-L looks to be mainly comprised of N705. There might be a bit of batch effect, so you could consider using batch correction on this dataset. However, if we focus our attention on the other cluster - mature T-cells -  where there is batch mixing, we can still assess this biologically even without batch correction.
Additionally, we will also look at the confounding effect of sex.</p>
<figure id="figure-17" style="max-width: 90%;"><img src="...
AAAASUVORK5CYII=
"" alt="Sex effect. " width="1623" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/sex_differences.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 17</strong>:</span> Sex differences</figcaption></figure>
<p>We note that the one female sample - unfortunately one of the mere three knockout samples - seems to be distributed in the same areas as the knockout samples at large, so luckily, this doesn’t seem to be a confounding factor and we can still learn from our data. Ideally, this experiment would be re-run with either more female samples all around or swapping out this female from the male sample.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-depth-effect"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Depth effect</div>
<p>Are there any clusters or differences being driven by sequencing depth, a technical and random factor?</p>
<figure id="figure-18" style="max-width: 90%;"><img src="...
BgwY/Lr4/wGs4iqnD4B2+gAAAABJRU5ErkJggg==
"" alt="Sequencing depth. " width="1110" height="421" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/counts_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 18</strong>:</span> Counts across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-13"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-13" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Eureka! This explains the odd DP shift between wildtype and knockout cells - the left side of the DP cells simply have a higher sequencing depth (UMIs/cell) than the ones on the right side. Well, that explains some of the sub-cluster that we’re seeing in that splurge. Importantly, we don’t see that the DP-L or (mostly) the mature T-cell clusters are similarly affected. So, whilst again, this variable of sequencing depth might be something to regress out somehow, it doesn’t seem to be impacting our dataset. The less you can regress/modify your data, in general, the better - you want to stay as true as you can to the raw data, and only use maths to correct your data when you really need to (and not to create insights where there are none!).</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-sample-purity"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Sample purity</div>
<p>Do you think we processed these samples well enough?</p>
<figure id="figure-19" style="max-width: 90%;"><img src="...
"" alt="Sequencing depth. " width="407" height="416" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/hemoglobin.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 19</strong>:</span> Hemoglobin across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-14"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-14" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>We have seen in the previous images that these clusters are not very tight or distinct, so we could consider stronger filtering. Additionally, hemoglobin - a red blood cell marker that should NOT be found in T-cells - appears throughout the entire sample in low numbers. This suggests some background in the media the cells were in, and we might consider in the wet lab trying to get a purer, happier sample, or in the dry lab, techniques such as SoupX or others to remove this background. Playing with filtering settings (increasing minimum counts/cell, etc.) is often the place to start in these scenarios.</p>
</blockquote>
</blockquote>
<blockquote class="question" style="border: 2px solid #8A9AD0; margin: 1em 0.2em">
<div class="box-title question-title" id="question-clustering-resolution"><i class="far fa-question-circle" aria-hidden="true" ></i> Question: Clustering resolution</div>
<p>Do you think the clustering is appropriate? i.e. are there single clusters that you think should be separate, and multiple clusters that could be combined?</p>
<figure id="figure-20" style="max-width: 90%;"><img src="...
rkJggg==
"" alt="Itm2a Expression. " width="1093" height="423" loading="lazy" /><a target="_blank" href="../../images/scrna-casestudy-jupyter/itm2a_across_clusters.png" rel="noopener noreferrer"><small>Open image in new tab</small></a><br /><br /><figcaption><span class="figcaption-prefix"><strong>Figure 20</strong>:</span> Itm2a across clusters</figcaption></figure>
<br/><details style="border: 2px solid #B8C3EA; margin: 1em 0.2em;padding: 0.5em; cursor: pointer;"><summary>👁 View solution</summary>
<div class="box-title solution-title" id="solution-15"><button class="gtn-boxify-button solution" type="button" aria-controls="solution-15" aria-expanded="true"><i class="far fa-eye" aria-hidden="true" ></i> <span>Solution</span><span class="fold-unfold fa fa-minus-square"></span></button></div>
<p>Important to note, lest all bioinformaticians combine forces to attack the biologists: just because a cluster doesn’t look like a cluster by eye is NOT enough to say it’s not a cluster! But looking at the biology here, we struggled to find marker genes to distinguish the DP population, which we know is also affected by depth of sequencing. That’s a reasonable argument that DP-M1, DP-M2, and DP-M3 might not be all that different. Maybe we need more depth of sequencing across all the DP cells, or to compare these explicitly to each other (consider variations on FindMarkers!). However, DP-L is both seemingly leaving the DP cluster and also has fewer knockout cells, so we might go and look at what DP-L is expressing in the marker genes. If we look at T-mat further, we can see that its marker gene - Itm2a - is only expressed in half of the cluster. You might consider sub-clustering this to investigate further, either through changing the resolution or through analysing this cluster alone.
If we look at the differences between genotypes alone (so the pseudo-bulk), we can see that most of the genes in that list are actually ribosomal. This might be a housekeeping background, this might be cell cycle related, this might be biological, or all three. You might consider investigating the cycling status of the cells, or even regressing this out (which is what the authors did).</p>
</blockquote>
</blockquote>
<p>Ultimately, there are quite a lot ways to analyse the data, both within the confines of this tutorial (the many parameters that could be changed throughout) and outside of it (batch correction, sub-clustering, cell-cycle scoring, inferred trajectories, etc.) Most analyses will still yield the same general output, though: there are fewer knockout cells in the mature T-cell population.</p>
<p>{% icon congratulations %} Congratulations! You have interpreted your plots in several important ways!</p>
<h1 id="export-your-data-and-notebook-and-figures">Export your data, and notebook, and figures</h1>
<p>It’s now time to export your data! First, we need to get Jupyter to see it as a file.</p>


In [ ]:
adata.write('MarkersCluster.h5ad')

<p>Now you can export it.</p>


In [ ]:
put("MarkersCluster.h5ad")

<p>To export your notebook to your Galaxy history, you can use the following. Change the text to be your notebook name. Do not use spaces!</p>


In [ ]:
put("name_of_jupyter_notebook.ipynb")

<p>Want to export some plots? Choose any (or all) of the plots you saved as files in the folder at the left and put their titles in the following. You can run multiple exports at the same time.</p>


In [ ]:
put("figures/plotname.png")
put("figures/plotname.png")
put("figures/plotname.png")

<h1 id="conclusion">Conclusion</h1>
<p>{% icon congratulations %} Congratulations! You’ve made it to the end!</p>
<p>In this tutorial, you moved from technical processing to biological exploration. By analysing real data - both the exciting and the messy! - you have, hopefully, experienced what it’s like to analyse and question a dataset, potentially without clear cut-offs or clear answers. If you were working in a group, you each analysed the data in different ways, and most likely found similar insights. One of the biggest problems in analysing scRNA-seq is the lack of a clearly defined pathway or parameters. You have to make the best call you can as you move through your analysis, and ultimately, when in doubt, try it multiple ways and see what happens!</p>
<p>If, for some reasons anything didn’t work in Galaxy JupyterLab environment, please don’t get discouraged - we prepared a <a href="https://colab.research.google.com/drive/1DkCysA77iaFAWoKJ1vwE5_qYV8Su7UsX?usp=sharing">Google Colab notebook version</a> for you as a backup so that you can enjoy the tutorial no matter what!</p>


# Key Points

- Single cell data is huge, and must have its many (# genes) dimensions reduced for analysis
- Analysis is more subjective than we think, and biological understanding of the samples as well as many iterations of analysis are important to give us our best change of attaining real biological insights

# Congratulations on successfully completing this tutorial!

Please [fill out the feedback on the GTN website](https://training.galaxyproject.org/training-material/topics/single-cell/tutorials/scrna-case-jupyter_basic-pipeline/tutorial.html#feedback) and check there for further resources!
